# KAGGLE CURRENT-WHEELHOUSE PREFLIGHT  (read-only; submits nothing; spends no submission slot)

**Purpose.** Compile the candidate archives against the wheelhouse that is ACTUALLY mounted in this Kaggle notebook (not the old
0.2.7 / 0.2.11 mirror), record exact versions, APIs and outputs, and replay one bounded conversation against a scripted
fake model to see what the agent sends.

**Status of the code below: written offline, never executed against the current wheelhouse** (the authoring environment
could not reach Kaggle). Anything that fails here is information; do not "fix" it by guessing. Every result is labelled
**CONFIRMED / INFERRED / UNKNOWN / BLOCKED / NOT TESTED** by what actually ran.

**Set up a NEW notebook (do not overwrite an existing one):**
1. Add data: the competition `gemma-4-developer-agent` and the dataset `metric/gemma-4-developer-agent-wheelhouse` (the organizer starter uses these).
2. Settings: Internet OFF (matches the scorer). An accelerator is not needed for compile or replay (UNVERIFIED; the optional evaluator smoke may want the same image as the starter).
3. **Compare the install cell below with the first code cell of the organizer's CURRENT starter notebook
   (`ryanholbrook/getting-started-gemma-4-developer-agent`). The copy this cell was based on is OLDER than the live one (UNVERIFIED difference). If they differ, paste theirs over cell 2.**
4. Optional: upload your own archives (for example the Prompt-6 "S0" zip) as a private dataset, attach it, and either list the path in `EXTRA_ZIPS` or let cell 5 auto-detect any `*.zip` with `agent.yaml` at its root.
5. Run all cells. Download `/kaggle/working/preflight_logs.zip` (also printed: the summary table). Nothing in the logs is secret.


In [ ]:
# ---- cell 1: configuration and helpers --------------------------------------------------------------------------------
import os, sys, json, hashlib, base64, subprocess, shutil, zipfile, glob, importlib, importlib.util, inspect, re, time, platform
import importlib.metadata as md
from pathlib import Path

WORK = Path(os.environ.get("NB_WORK", "/kaggle/working"))
LOGS = WORK / "logs"; LOGS.mkdir(parents=True, exist_ok=True)
TOOLS = WORK / "tools"; TOOLS.mkdir(parents=True, exist_ok=True)
EXTRACT = WORK / "extracted"; EXTRACT.mkdir(parents=True, exist_ok=True)
INSTALL_MODE = os.environ.get("NB_INSTALL_MODE", "official")    # "official" = organizer-starter procedure; "skip" = harness already importable
HARNESS_DIR = os.environ.get("NB_HARNESS_DIR", "-")             # "-" = wheels are pip-installed
WHEELHOUSE_DIR = Path("/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse")
DATA_DIR = Path("/kaggle/input/competitions/gemma-4-developer-agent")
EXTRA_ZIPS = []                          # e.g. ["/kaggle/input/my-candidates/s0.zip"]
RUN_OFFICIAL_EVALUATOR_SMOKE = False     # opt-in: runs the real Evaluator on ONE task against the scripted fake model
MODEL = "gemma-4-31b-it-qat-w4a16-ct"
RESULTS = {}                             # label -> dict of what actually ran

if HARNESS_DIR != "-":
    sys.path.insert(0, HARNESS_DIR)
os.environ.setdefault("LITELLM_LOCAL_MODEL_COST_MAP", "True")

def log(name, text):
    (LOGS / name).write_text(text if isinstance(text, str) else json.dumps(text, indent=2, default=str), encoding="utf-8")

def sha256_file(p, chunk=1 << 20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

def tail(s, n=1500):
    return s[-n:] if s else ""

print("python", sys.version.split()[0], "|", platform.platform())
print("WORK", WORK, "| INSTALL_MODE", INSTALL_MODE, "| HARNESS_DIR", HARNESS_DIR)


In [ ]:
# ---- cell 2: offline wheelhouse install (same procedure as the organizer starter; REPLACE with their current cell if it differs) ---
if INSTALL_MODE == "skip":
    print("INSTALL SKIPPED (INSTALL_MODE=skip): using whatever is importable. Not a Kaggle-wheelhouse run.")
    log("00_install.log", "INSTALL SKIPPED (local test mode)")
else:
    out = []
    if not WHEELHOUSE_DIR.exists():
        hits = sorted({str(Path(p).parent) for p in glob.glob("/kaggle/input/**/swegemma*.whl", recursive=True)})
        out.append(f"default wheelhouse path missing; rglob found: {hits}")
        if hits:
            WHEELHOUSE_DIR = Path(hits[0])
    out.append(f"WHEELHOUSE_DIR = {WHEELHOUSE_DIR} exists={WHEELHOUSE_DIR.exists()}")
    if not WHEELHOUSE_DIR.exists():
        out.append("BLOCKED: wheelhouse not mounted. Add dataset metric/gemma-4-developer-agent-wheelhouse to this notebook.")
    else:
        for pat in ("/usr/local/lib/python*/dist-packages/*cutlass*.pth", "/usr/local/lib/python*/site-packages/*cutlass*.pth"):
            for pth in glob.glob(pat):
                try: os.unlink(pth); out.append(f"removed {pth}")
                except OSError: pass
        tmp = Path("/tmp/wheelhouse"); tmp.mkdir(parents=True, exist_ok=True)
        for w in WHEELHOUSE_DIR.glob("*.whl"):
            if "cutlass" in w.name.lower():
                continue
            name = w.name.replace("cu128", "+cu128") if ("cu128" in w.name and "+" not in w.name) else w.name
            if not (tmp / name).exists():
                os.symlink(w, tmp / name)
        wheels = sorted(str(w) for w in tmp.glob("*.whl"))
        out.append(f"installing {len(wheels)} wheels")
        t0 = time.time()
        p = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--force-reinstall", *wheels],
                           capture_output=True, text=True)
        out.append(f"pip exit code {p.returncode} after {time.time()-t0:.0f}s")
        out.append("pip stderr tail: " + tail(p.stderr, 2000))
        importlib.invalidate_caches()
    log("00_install.log", "\n".join(out)); print("\n".join(out))


In [ ]:
# ---- cell 3: versions, wheel inventory, scorer scripts ---------------------------------------------------------------------
disc = {"python": sys.version, "platform": platform.platform(), "versions": {}, "wheel_inventory": [], "scorer_script_hits": []}
for d in ("swegemma", "adk-submission", "adk-eval-core", "google-adk", "google-genai", "litellm", "pydantic", "vllm", "torch", "pyyaml"):
    try: disc["versions"][d] = md.version(d)
    except Exception as e: disc["versions"][d] = f"NOT INSTALLED ({type(e).__name__})"
if WHEELHOUSE_DIR.exists():
    for w in sorted(WHEELHOUSE_DIR.glob("*.whl")):
        small = w.stat().st_size < 50_000_000
        disc["wheel_inventory"].append({"file": w.name, "bytes": w.stat().st_size, "sha256": sha256_file(w) if small else "(not hashed: >50MB)"})
for pat in ("/kaggle/input/**/inference.py", "/kaggle/input/**/metric.py", "/kaggle/input/**/scripts/*.py"):
    disc["scorer_script_hits"] += glob.glob(pat, recursive=True)[:20]
for extra in ("datapackage.json", "dataset-metadata.json"):
    q = WHEELHOUSE_DIR / extra
    if q.exists():
        disc[extra] = q.read_text()[:3000]
disc["competition_dir_listing"] = sorted(p.name for p in DATA_DIR.iterdir())[:50] if DATA_DIR.exists() else "NOT MOUNTED"
log("01_discovery.json", disc)
print(json.dumps({k: disc[k] for k in ("versions", "scorer_script_hits", "competition_dir_listing")}, indent=1))
print(f"{len(disc['wheel_inventory'])} wheels in the wheelhouse; full list in logs/01_discovery.json")


In [ ]:
# ---- cell 4: what the INSTALLED packages actually expose (no assumptions) ---------------------------------------------------
api = {}
def probe(label, fn):
    try: api[label] = fn()
    except BaseException as e: api[label] = f"ABSENT/ERROR: {type(e).__name__}: {e}"
def sig(obj): return str(inspect.signature(obj))
try:
    import adk_submission, swegemma
    IMPORT_OK = True
except BaseException as e:
    IMPORT_OK = False; adk_submission = swegemma = None
    api["IMPORT_ERROR"] = f"{type(e).__name__}: {e}"
    print("BLOCKED: harness import failed ->", api["IMPORT_ERROR"])
probe("adk_submission.__file__", lambda: adk_submission.__file__)
probe("swegemma.__file__", lambda: swegemma.__file__)
probe("adk_submission.compile_submission", lambda: sig(adk_submission.compile_submission))
probe("adk_submission.discover_adapters", lambda: sig(adk_submission.discover_adapters))
probe("adk_submission.discovery.validate_directory", lambda: sig(importlib.import_module("adk_submission.discovery").validate_directory))
probe("adk_submission.discovery.discover_declared_models", lambda: sig(importlib.import_module("adk_submission.discovery").discover_declared_models))
probe("adk_submission.ModelRegistry.register", lambda: sig(adk_submission.ModelRegistry.register))
probe("adk_submission.ToolRegistry.register", lambda: sig(adk_submission.ToolRegistry.register))
probe("swegemma.config.build_submission_limits", lambda: sig(importlib.import_module("swegemma.config").build_submission_limits))
probe("swegemma.config.constants", lambda: {k: str(getattr(importlib.import_module("swegemma.config"), k)) for k in
      ("ALLOWED_SUBMISSION_EXTENSIONS", "ALLOWED_ADAPTER_EXTENSIONS", "MAX_SUBMISSION_SIZE_BYTES") if hasattr(importlib.import_module("swegemma.config"), k)})
probe("swegemma.context.SwegemmaContext", lambda: sig(importlib.import_module("swegemma.context").SwegemmaContext.__init__))
for modname, attr in [("swegemma.models", "validate_single_declared_model"), ("swegemma.models.discovery", "validate_single_declared_model"),
                      ("swegemma.models.registry", "setup_gemma_model_registry"), ("swegemma.models", "load_tasks"),
                      ("swegemma.cli", "_build_cli_model_registry"), ("swegemma.harness.agent_runner", "resolve_swegemma_adapter"),
                      ("swegemma.models", "resolve_swegemma_adapter"), ("swegemma.evaluate", "Evaluator"), ("swegemma.config", "EvalConfig")]:
    probe(f"{modname}.{attr}", lambda m=modname, a=attr: "PRESENT " + (sig(getattr(importlib.import_module(m), a)) if callable(getattr(importlib.import_module(m), a)) else "(non-callable)"))
def evalconfig_fields():
    C = importlib.import_module("swegemma.config").EvalConfig
    return sorted(getattr(C, "model_fields", None) or getattr(C, "__dataclass_fields__", {}) or [])
probe("swegemma.config.EvalConfig fields", evalconfig_fields)
# the thinking-config bridge: find the module(s) that mention enable_thinking and keep the relevant lines verbatim
bridge = []
for root in ({Path(adk_submission.__file__).parent, Path(swegemma.__file__).parent} if IMPORT_OK else set()):
    for f in sorted(root.rglob("*.py")):
        try: lines = f.read_text(encoding="utf-8").splitlines()
        except Exception: continue
        for i, line in enumerate(lines, 1):
            if re.search(r"enable_thinking|thinking_level|thinking_budget|include_thoughts", line):
                bridge.append(f"{f}:{i}: {line.strip()}")
log("10_api_discovery.json", api)
log("11_thinking_bridge_lines.txt", "\n".join(bridge) or "NO MATCH FOUND")
print(json.dumps(api, indent=1, default=str)[:6000])
print(f"\n{len(bridge)} source lines mention thinking settings (logs/11_thinking_bridge_lines.txt)")


In [ ]:
# ---- cell 5: write the embedded tools + candidate archives (SHA-256 verified), find extra archives, extract each into a fresh dir ---
EMBED = {
    "preflight_compile.py": ("fd9c4474556260f049d509cc788a637346b71f990af19b037e5a60a8f1d6c856", (
        "IiIiUFJFLUZMSUdIVCBDT01QSUxFIG9mIGFuIHVucGFja2VkIHN1Ym1pc3Npb24gZGlyZWN0b3J5IGFnYWluc3QgdGhlIGhhcm5l"
        "c3Mgd2hlZWxzIHRoYXQgYXJlIElOU1RBTExFRAoob3IgdW5wYWNrZWQgaW4gPGhhcm5lc3NfZGlyPikuICBWZXJzaW9uLXRvbGVy"
        "YW50OiBoZWxwZXIgaW1wb3J0cyB0aGF0IG1vdmVkIGJldHdlZW4gc3dlZ2VtbWEgcmVsZWFzZXMgZmFsbCBiYWNrIHRvCndoYXQg"
        "YWRrX3N1Ym1pc3Npb24gaXRzZWxmIHByb3ZpZGVzLgoKICAgIHB5dGhvbiAtSSBwcmVmbGlnaHRfY29tcGlsZS5weSA8aGFybmVz"
        "c19kaXIgfCAtPiA8c3VibWlzc2lvbl9kaXI+CgogIDxoYXJuZXNzX2Rpcj4gIGRpcmVjdG9yeSBob2xkaW5nIHVucGFja2VkIHdo"
        "ZWVscyAoYWRrX3N1Ym1pc3Npb24vLCBzd2VnZW1tYS8sIGFka19ldmFsX2NvcmUvIGFuZCB0aGVpciAqLmRpc3QtaW5mbyksCiAg"
        "ICAgICAgICAgICAgICAgb3IgJy0nIHdoZW4gdGhlIHdoZWVscyBhcmUgYWxyZWFkeSBwaXAtaW5zdGFsbGVkIChlLmcuIGluIGEg"
        "S2FnZ2xlIG5vdGVib29rIHdpdGggdGhlIHdoZWVsaG91c2UgYXR0YWNoZWQpLgogIDxzdWJtaXNzaW9uX2Rpcj4gIHRoZSBVTlpJ"
        "UFBFRCBzdWJtaXNzaW9uIChhZ2VudC55YW1sIGF0IGl0cyB0b3AgbGV2ZWwpLiBVbnppcCBpbnRvIGEgZnJlc2ggZW1wdHkgZm9s"
        "ZGVyIGZpcnN0LgoKUHJpbnRzIE9ORSBKU09OIG9iamVjdDogaW5zdGFsbGVkIHZlcnNpb25zLCBvbmUgZW50cnkgcGVyIHN0ZXAs"
        "IFZFUkRJQ1RfY29tcGlsZV9zdWJtaXNzaW9uID0gUEFTUyB8IEZBSUwuCkFsd2F5cyBleGl0cyAwOyB0aGUgdmVyZGljdCBpcyBp"
        "biB0aGUgSlNPTi4gIFNldCBQUkVGTElHSFRfRk9SQ0VfRkFMTEJBQ0s9MSB0byBleGVyY2lzZSB0aGUgZmFsbGJhY2sgY29kZSBw"
        "YXRocy4KCldIQVQgQSBQQVNTIE1FQU5TOiB0aGUgb3JnYW5pemVycycgWUFNTCBsb2FkZXIsIHNjaGVtYSwgdG9vbC9tb2RlbCBy"
        "ZWdpc3RyaWVzIGFuZCBnZW5lcmF0aW9uIGNvbnN0cmFpbnRzLCBpbiB0aGUKaW5zdGFsbGVkIHZlcnNpb24sIGFjY2VwdCB0aGUg"
        "ZGlyZWN0b3J5LiAgV0hBVCBJVCBET0VTIE5PVCBNRUFOOiBub3RoaW5nIGhlcmUgc3RhcnRzIHZMTE0sIHJ1bnMgR2VtbWEsIHVz"
        "ZXMgYSBzYW5kYm94LApvciBydW5zIHRoZSByZWFsIHNjb3JlciAoc2NyaXB0cy9pbmZlcmVuY2UucHkgaXMgbm90IHB1YmxpYyku"
        "ICBGYWxsYmFjayBtb2RlIHVzZXMgc3R1YiB0b29scyBhbmQgYSBzdHViIG1vZGVsLCBzbyB0aGUKdGhpbmtpbmctY29uZmlnIGJy"
        "aWRnZSBpcyBub3QgZXhlcmNpc2VkLgoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGltcG9y"
        "dGxpYi5tZXRhZGF0YSBhcyBtZAppbXBvcnQganNvbgppbXBvcnQgb3MKaW1wb3J0IHN5cwppbXBvcnQgdHJhY2ViYWNrCmZyb20g"
        "cGF0aGxpYiBpbXBvcnQgUGF0aAoKaGFybmVzc19kaXIsIHN1Yl9kaXIgPSBzeXMuYXJndlsxXSwgUGF0aChzeXMuYXJndlsyXSkK"
        "aWYgaGFybmVzc19kaXIgIT0gIi0iOgogICAgc3lzLnBhdGguaW5zZXJ0KDAsIGhhcm5lc3NfZGlyKQpvcy5lbnZpcm9uLnNldGRl"
        "ZmF1bHQoIkxJVEVMTE1fTE9DQUxfTU9ERUxfQ09TVF9NQVAiLCAiVHJ1ZSIpCkZPUkNFX0ZBTExCQUNLID0gb3MuZW52aXJvbi5n"
        "ZXQoIlBSRUZMSUdIVF9GT1JDRV9GQUxMQkFDSyIpID09ICIxIgpNT0RFTCA9ICJnZW1tYS00LTMxYi1pdC1xYXQtdzRhMTYtY3Qi"
        "CgpyZXBvcnQ6IGRpY3QgPSB7InN1Ym1pc3Npb25fZGlyIjogc3RyKHN1Yl9kaXIpLCAiaGFybmVzc19kaXIiOiBoYXJuZXNzX2Rp"
        "ciwgImZvcmNlX2ZhbGxiYWNrIjogRk9SQ0VfRkFMTEJBQ0ssCiAgICAgICAgICAgICAgICAidmVyc2lvbnMiOiB7fSwgInN0ZXBz"
        "IjogW10sICJub3RlcyI6IFtdfQpmb3IgZGlzdCBpbiAoInN3ZWdlbW1hIiwgImFkay1zdWJtaXNzaW9uIiwgImFkay1ldmFsLWNv"
        "cmUiLCAiZ29vZ2xlLWFkayIpOgogICAgdHJ5OgogICAgICAgIHJlcG9ydFsidmVyc2lvbnMiXVtkaXN0XSA9IG1kLnZlcnNpb24o"
        "ZGlzdCkKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZTogICMgbm9xYTogQkxFMDAxCiAgICAgICAgcmVwb3J0WyJ2ZXJzaW9ucyJd"
        "W2Rpc3RdID0gZiJVTktOT1dOICh7dHlwZShlKS5fX25hbWVfX30pIgoKCmRlZiBzdGVwKG5hbWUsIGZuKToKICAgIHRyeToKICAg"
        "ICAgICB2YWwgPSBmbigpCiAgICAgICAgcmVwb3J0WyJzdGVwcyJdLmFwcGVuZCh7InN0ZXAiOiBuYW1lLCAib2siOiBUcnVlLCAi"
        "ZGV0YWlsIjogdmFsfSkKICAgICAgICByZXR1cm4gVHJ1ZSwgdmFsCiAgICBleGNlcHQgQmFzZUV4Y2VwdGlvbiBhcyBlOiAgIyBu"
        "b3FhOiBCTEUwMDEgLSByZWNvcmQgZXZlcnkgZmFpbHVyZQogICAgICAgIHJlcG9ydFsic3RlcHMiXS5hcHBlbmQoeyJzdGVwIjog"
        "bmFtZSwgIm9rIjogRmFsc2UsICJleGNfdHlwZSI6IGYie3R5cGUoZSkuX19tb2R1bGVfX30ue3R5cGUoZSkuX19uYW1lX199IiwK"
        "ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAibWVzc2FnZSI6IHN0cihlKSwgInRyYWNlYmFja190YWlsIjogdHJhY2Vi"
        "YWNrLmZvcm1hdF9leGMoKS5zdHJpcCgpLnNwbGl0bGluZXMoKVstMTA6XX0pCiAgICAgICAgcmV0dXJuIEZhbHNlLCBOb25lCgoK"
        "IyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIGltcG9ydHMg"
        "dGhhdCBleGlzdCBpbiBldmVyeSByZWxlYXNlIHNlZW4gc28gZmFyCmZyb20gYWRrX3N1Ym1pc3Npb24gaW1wb3J0IFRvb2xSZWdp"
        "c3RyeSwgY29tcGlsZV9zdWJtaXNzaW9uICAjIG5vcWE6IEU0MDIKZnJvbSBhZGtfc3VibWlzc2lvbi5kaXNjb3ZlcnkgaW1wb3J0"
        "IHZhbGlkYXRlX2RpcmVjdG9yeSAgIyBub3FhOiBFNDAyCmZyb20gc3dlZ2VtbWEuY29uZmlnIGltcG9ydCBidWlsZF9zdWJtaXNz"
        "aW9uX2xpbWl0cyAgIyBub3FhOiBFNDAyCgpsaW1pdHMsIGdlbl9jb25zdHJhaW50cyA9IGJ1aWxkX3N1Ym1pc3Npb25fbGltaXRz"
        "KCkKdHJ5OgogICAgZnJvbSBzd2VnZW1tYS5jb25maWcgaW1wb3J0IEFMTE9XRURfQURBUFRFUl9FWFRFTlNJT05TICAjIG5vcWE6"
        "IEU0MDIKZXhjZXB0IEltcG9ydEVycm9yOgogICAgQUxMT1dFRF9BREFQVEVSX0VYVEVOU0lPTlMgPSBmcm96ZW5zZXQoeyIuc2Fm"
        "ZXRlbnNvcnMifSkKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0t"
        "LS0tLS0gc3RlcCAxOiBkaXJlY3Rvcnkgc3RydWN0dXJlCnN0ZXAoInZhbGlkYXRlX2RpcmVjdG9yeSIsIGxhbWJkYTogc3RyKHZh"
        "bGlkYXRlX2RpcmVjdG9yeShzdWJfZGlyLCBsaW1pdHMpLmNvbmZpZ19wYXRoKSkKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0t"
        "LS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gc3RlcCAyOiBleGFjdGx5IG9uZSBkZWNsYXJlZCBtb2Rl"
        "bCA9PSBjb21wZXRpdGlvbiBtb2RlbApkZWYgc2luZ2xlX21vZGVsKCk6CiAgICBmbiA9IE5vbmUKICAgIGlmIG5vdCBGT1JDRV9G"
        "QUxMQkFDSzoKICAgICAgICBmb3IgbW9kIGluICgic3dlZ2VtbWEubW9kZWxzIiwgInN3ZWdlbW1hLm1vZGVscy5kaXNjb3Zlcnki"
        "KToKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgZm4gPSBnZXRhdHRyKF9faW1wb3J0X18obW9kLCBmcm9tbGlzdD1b"
        "IngiXSksICJ2YWxpZGF0ZV9zaW5nbGVfZGVjbGFyZWRfbW9kZWwiKQogICAgICAgICAgICAgICAgcmVwb3J0WyJub3RlcyJdLmFw"
        "cGVuZChmInNpbmdsZS1tb2RlbCBjaGVjazogc3dlZ2VtbWEgaGVscGVyIGZyb20ge21vZH0iKQogICAgICAgICAgICAgICAgYnJl"
        "YWsKICAgICAgICAgICAgZXhjZXB0IChJbXBvcnRFcnJvciwgQXR0cmlidXRlRXJyb3IpOgogICAgICAgICAgICAgICAgY29udGlu"
        "dWUKICAgIGlmIGZuIGlzIG5vdCBOb25lOgogICAgICAgIHJldHVybiBmbihzdWJfZGlyKQogICAgZnJvbSBhZGtfc3VibWlzc2lv"
        "bi5kaXNjb3ZlcnkgaW1wb3J0IGRpc2NvdmVyX2RlY2xhcmVkX21vZGVscyAgIyBmYWxsYmFjayAodXNlZCBieSB0aGUgMC4yLjEx"
        "IG9mZmljaWFsIG5vdGVib29rKQogICAgdHJ5OgogICAgICAgIG5hbWVzID0gZGlzY292ZXJfZGVjbGFyZWRfbW9kZWxzKHN0cihz"
        "dWJfZGlyKSkKICAgIGV4Y2VwdCBUeXBlRXJyb3I6CiAgICAgICAgbmFtZXMgPSBkaXNjb3Zlcl9kZWNsYXJlZF9tb2RlbHMoc3Ry"
        "KHN1Yl9kaXIpLCBOb25lKQogICAgbmFtZXMgPSB7c3RyKG4pLnJlbW92ZXByZWZpeCgib3BlbmFpLyIpIGZvciBuIGluIG5hbWVz"
        "fQogICAgcmVwb3J0WyJub3RlcyJdLmFwcGVuZCgic2luZ2xlLW1vZGVsIGNoZWNrOiBGQUxMQkFDSyBhZGtfc3VibWlzc2lvbi5k"
        "aXNjb3ZlcnkuZGlzY292ZXJfZGVjbGFyZWRfbW9kZWxzIikKICAgIGlmIG5hbWVzICE9IHtNT0RFTH06CiAgICAgICAgcmFpc2Ug"
        "VmFsdWVFcnJvcihmImRlY2xhcmVkIG1vZGVscyB7c29ydGVkKG5hbWVzKX0gIT0gZXhhY3RseSBbe01PREVMIXJ9XSIpCiAgICBy"
        "ZXR1cm4gTU9ERUwKCgpzdGVwKCJzaW5nbGVfZGVjbGFyZWRfbW9kZWwiLCBzaW5nbGVfbW9kZWwpCgoKIyAtLS0tLS0tLS0tLS0t"
        "LS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIHN0ZXAgMzogYWRhcHRlcnMKZGVmIGFk"
        "YXB0ZXJzKCk6CiAgICBmcm9tIGFka19zdWJtaXNzaW9uIGltcG9ydCBkaXNjb3Zlcl9hZGFwdGVycwogICAgdHJ5OgogICAgICAg"
        "IG0gPSBkaXNjb3Zlcl9hZGFwdGVycyhzdHIoc3ViX2RpciksIGFkYXB0ZXJfZXh0ZW5zaW9ucz1BTExPV0VEX0FEQVBURVJfRVhU"
        "RU5TSU9OUykKICAgIGV4Y2VwdCBUeXBlRXJyb3I6CiAgICAgICAgbSA9IGRpc2NvdmVyX2FkYXB0ZXJzKHN0cihzdWJfZGlyKSkK"
        "ICAgIHJldHVybiBzb3J0ZWQoZ2V0YXR0cihtLCAiYWRhcHRlcnMiLCB7fSkua2V5cygpKSwgbQoKCm9rX2FkLCBhZCA9IHN0ZXAo"
        "ImRpc2NvdmVyX2FkYXB0ZXJzIiwgYWRhcHRlcnMpCm1hbmlmZXN0ID0gTm9uZQppZiBva19hZDoKICAgIHJlcG9ydFsic3RlcHMi"
        "XVstMV1bImRldGFpbCJdLCBtYW5pZmVzdCA9IGFkCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0t"
        "LS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIHN0ZXAgNDogZnVsbCBjb21waWxlCmRlZiBidWlsZF90b29scygpOgogICAgaWYgbm90"
        "IEZPUkNFX0ZBTExCQUNLOgogICAgICAgIHRyeToKICAgICAgICAgICAgZnJvbSBzd2VnZW1tYS50b29scyBpbXBvcnQgY3JlYXRl"
        "X3Rvb2xzCiAgICAgICAgICAgIHJldHVybiBkaWN0KGNyZWF0ZV90b29scyhOb25lKSksICJzd2VnZW1tYS50b29scy5jcmVhdGVf"
        "dG9vbHMoTm9uZSkiCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjogICMgbm9xYTogQkxFMDAxCiAgICAgICAgICAgIHBhc3MKICAg"
        "ICAgICB0cnk6CiAgICAgICAgICAgIGZyb20gc3dlZ2VtbWEuY29udGV4dCBpbXBvcnQgU3dlZ2VtbWFDb250ZXh0CiAgICAgICAg"
        "ICAgIHJldHVybiBkaWN0KFN3ZWdlbW1hQ29udGV4dChwcm9ibGVtX3N0YXRlbWVudD0ieCIsIHRhc2tfaWQ9InQiLCByZXBvPSJy"
        "IikuY3JlYXRlX3Rvb2xzKCkpLCAiU3dlZ2VtbWFDb250ZXh0LmNyZWF0ZV90b29scygpIgogICAgICAgIGV4Y2VwdCBFeGNlcHRp"
        "b246ICAjIG5vcWE6IEJMRTAwMQogICAgICAgICAgICBwYXNzCgogICAgZGVmIHJ1bl9jb21tYW5kKGNvbW1hbmQ6IHN0cikgLT4g"
        "c3RyOiByZXR1cm4gIiIKICAgIGRlZiByZWFkX2ZpbGUoZmlsZXBhdGg6IHN0ciwgc3RhcnRfbGluZTogaW50IHwgTm9uZSA9IE5v"
        "bmUsIGVuZF9saW5lOiBpbnQgfCBOb25lID0gTm9uZSkgLT4gc3RyOiByZXR1cm4gIiIKICAgIGRlZiB3cml0ZV9maWxlKGZpbGVw"
        "YXRoOiBzdHIsIGNvbnRlbnQ6IHN0cikgLT4gc3RyOiByZXR1cm4gIiIKICAgIGRlZiBlZGl0X2ZpbGUoZmlsZXBhdGg6IHN0ciwg"
        "b2xkX3N0cmluZzogc3RyLCBuZXdfc3RyaW5nOiBzdHIsIGFsbG93X211bHRpcGxlOiBib29sID0gRmFsc2UpIC0+IHN0cjogcmV0"
        "dXJuICIiCiAgICBkZWYgZ2V0X3N0YXR1cygpIC0+IHN0cjogcmV0dXJuICIiCiAgICBkZWYgc3VibWl0X3BhdGNoKCkgLT4gc3Ry"
        "OiByZXR1cm4gIiIKICAgIGRlZiBnZXRfY29kZV9uZWlnaGJvcnMobm9kZTogc3RyLCBlZGdlX3R5cGU6IHN0ciB8IE5vbmUgPSBO"
        "b25lLCBtYXhfbmVpZ2hib3JzOiBpbnQgPSA1MCkgLT4gc3RyOiByZXR1cm4gIiIKICAgIGRlZiBzZWFyY2hfc2ltaWxhcl9jb2Rl"
        "KHF1ZXJ5OiBzdHIsIGs6IGludCA9IDEwKSAtPiBzdHI6IHJldHVybiAiIgogICAgZGVmIGdldF9jb2RlX3N1YmdyYXBoKG5vZGVz"
        "OiBsaXN0W3N0cl0pIC0+IHN0cjogcmV0dXJuICIiCiAgICBzdHVicyA9IHtmLl9fbmFtZV9fOiBmIGZvciBmIGluIChydW5fY29t"
        "bWFuZCwgcmVhZF9maWxlLCB3cml0ZV9maWxlLCBlZGl0X2ZpbGUsIGdldF9zdGF0dXMsIHN1Ym1pdF9wYXRjaCwKICAgICAgICAg"
        "ICAgICAgICAgICAgICAgICAgICAgICAgICAgIGdldF9jb2RlX25laWdoYm9ycywgc2VhcmNoX3NpbWlsYXJfY29kZSwgZ2V0X2Nv"
        "ZGVfc3ViZ3JhcGgpfQogICAgcmV0dXJuIHN0dWJzLCAiRkFMTEJBQ0sgc3R1YiB0b29scyAobmFtZXMgKyBzaWduYXR1cmVzIG9u"
        "bHkpIgoKCmRlZiBidWlsZF9tb2RlbHMoKToKICAgIGlmIG5vdCBGT1JDRV9GQUxMQkFDSzoKICAgICAgICBmb3IgbW9kIGluICgi"
        "c3dlZ2VtbWEubW9kZWxzLnJlZ2lzdHJ5IiwgInN3ZWdlbW1hLm1vZGVscyIpOgogICAgICAgICAgICB0cnk6CiAgICAgICAgICAg"
        "ICAgICBzZXR1cCA9IGdldGF0dHIoX19pbXBvcnRfXyhtb2QsIGZyb21saXN0PVsieCJdKSwgInNldHVwX2dlbW1hX21vZGVsX3Jl"
        "Z2lzdHJ5IikKICAgICAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgICAgICByZWcgPSBzZXR1cChhcGlfYmFzZT0iaHR0"
        "cDovLzEyNy4wLjAuMTo4MDAwL3YxIiwgc2VydmVkX21vZGVsPU1PREVMLCBhZGFwdGVyX21hbmlmZXN0PW1hbmlmZXN0KQogICAg"
        "ICAgICAgICAgICAgZXhjZXB0IFR5cGVFcnJvcjoKICAgICAgICAgICAgICAgICAgICByZWcgPSBzZXR1cChhcGlfYmFzZT0iaHR0"
        "cDovLzEyNy4wLjAuMTo4MDAwL3YxIiwgYXBpX2tleT0iRU1QVFkiKQogICAgICAgICAgICAgICAgcmVzb2x2ZXIgPSBOb25lCiAg"
        "ICAgICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICAgICAgZnJvbSBzd2VnZW1tYS5tb2RlbHMucmVnaXN0cnkgaW1wb3J0"
        "IHJlc29sdmVfc3dlZ2VtbWFfYWRhcHRlciBhcyByCiAgICAgICAgICAgICAgICAgICAgcmVzb2x2ZXIgPSBsYW1iZGEgYiwgaTog"
        "cihiLCBpLCByZWcpICAjIG5vcWE6IEU3MzEKICAgICAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb246ICAjIG5vcWE6IEJMRTAw"
        "MQogICAgICAgICAgICAgICAgICAgIHBhc3MKICAgICAgICAgICAgICAgIHJldHVybiByZWcsIHJlc29sdmVyLCBmInttb2R9LnNl"
        "dHVwX2dlbW1hX21vZGVsX3JlZ2lzdHJ5IgogICAgICAgICAgICBleGNlcHQgKEltcG9ydEVycm9yLCBBdHRyaWJ1dGVFcnJvcik6"
        "CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgZnJvbSBhZGtfc3VibWlzc2lvbiBpbXBvcnQgTW9kZWxSZWdpc3RyeQogICAg"
        "ZnJvbSBnb29nbGUuYWRrLm1vZGVscy5iYXNlX2xsbSBpbXBvcnQgQmFzZUxsbQoKICAgIGNsYXNzIF9TdHViTGxtKEJhc2VMbG0p"
        "OgogICAgICAgIG1vZGVsOiBzdHIgPSAic3R1Yi1jb21waWxlLW9ubHkiCgogICAgICAgIEBjbGFzc21ldGhvZAogICAgICAgIGRl"
        "ZiBzdXBwb3J0ZWRfbW9kZWxzKGNscyk6CiAgICAgICAgICAgIHJldHVybiBbciIuKiJdCgogICAgICAgIGFzeW5jIGRlZiBnZW5l"
        "cmF0ZV9jb250ZW50X2FzeW5jKHNlbGYsIGxsbV9yZXF1ZXN0LCBzdHJlYW09RmFsc2UpOgogICAgICAgICAgICByYWlzZSBSdW50"
        "aW1lRXJyb3IoInN0dWIgbW9kZWw6IGNvbXBpbGUtb25seSIpCiAgICAgICAgICAgIHlpZWxkICAjIG5vcWE6IHVucmVhY2hhYmxl"
        "IC0gbWFrZXMgdGhpcyBhbiBhc3luYyBnZW5lcmF0b3IKCiAgICByZWcgPSBNb2RlbFJlZ2lzdHJ5KCkKICAgIHJlZy5yZWdpc3Rl"
        "cihNT0RFTCwgX1N0dWJMbG0oKSkKICAgIHJldHVybiByZWcsIE5vbmUsICJGQUxMQkFDSyBhZGtfc3VibWlzc2lvbi5Nb2RlbFJl"
        "Z2lzdHJ5ICsgc3R1YiBtb2RlbCIKCgpkZWYgZGVzY3JpYmUoYWdlbnQpOgogICAgZ2NjID0gZ2V0YXR0cihhZ2VudCwgImdlbmVy"
        "YXRlX2NvbnRlbnRfY29uZmlnIiwgTm9uZSkKICAgIHJldHVybiB7CiAgICAgICAgIm5hbWUiOiBhZ2VudC5uYW1lLCAidHlwZSI6"
        "IHR5cGUoYWdlbnQpLl9fbmFtZV9fLAogICAgICAgICJtb2RlbCI6IGdldGF0dHIoZ2V0YXR0cihhZ2VudCwgIm1vZGVsIiwgTm9u"
        "ZSksICJtb2RlbCIsIHN0cihnZXRhdHRyKGFnZW50LCAibW9kZWwiLCBOb25lKSkpLAogICAgICAgICJpbnN0cnVjdGlvbl9jaGFy"
        "cyI6IGxlbihhZ2VudC5pbnN0cnVjdGlvbikgaWYgaXNpbnN0YW5jZShnZXRhdHRyKGFnZW50LCAiaW5zdHJ1Y3Rpb24iLCBOb25l"
        "KSwgc3RyKSBlbHNlIE5vbmUsCiAgICAgICAgInRvb2xzIjogW2dldGF0dHIodCwgIm5hbWUiLCBnZXRhdHRyKHQsICJfX25hbWVf"
        "XyIsIHR5cGUodCkuX19uYW1lX18pKSBmb3IgdCBpbiBnZXRhdHRyKGFnZW50LCAidG9vbHMiLCBbXSldLAogICAgICAgICJnZW5l"
        "cmF0ZV9jb250ZW50X2NvbmZpZyI6IGdjYy5tb2RlbF9kdW1wKGV4Y2x1ZGVfbm9uZT1UcnVlKSBpZiBnY2MgaXMgbm90IE5vbmUg"
        "ZWxzZSBOb25lLAogICAgICAgICJzdWJfYWdlbnRzIjogW2Rlc2NyaWJlKHMpIGZvciBzIGluIGdldGF0dHIoYWdlbnQsICJzdWJf"
        "YWdlbnRzIiwgW10pXSwKICAgICAgICAiYWdlbnRfdG9vbHMiOiBbeyJ0b29sIjogZ2V0YXR0cih0LCAibmFtZSIsICI/IiksICJz"
        "a2lwX3N1bW1hcml6YXRpb24iOiBnZXRhdHRyKHQsICJza2lwX3N1bW1hcml6YXRpb24iLCBOb25lKSwKICAgICAgICAgICAgICAg"
        "ICAgICAgICAgICJpbm5lciI6IGRlc2NyaWJlKHQuYWdlbnQpfSBmb3IgdCBpbiBnZXRhdHRyKGFnZW50LCAidG9vbHMiLCBbXSkg"
        "aWYgaGFzYXR0cih0LCAiYWdlbnQiKV0sCiAgICB9CgoKZGVmIGRvX2NvbXBpbGUoKToKICAgIHRvb2xzLCB0b29sc19zcmMgPSBi"
        "dWlsZF90b29scygpCiAgICBtb2RlbHMsIHJlc29sdmVyLCBtb2RlbHNfc3JjID0gYnVpbGRfbW9kZWxzKCkKICAgIHJlcG9ydFsi"
        "bm90ZXMiXS5hcHBlbmQoZiJjb21waWxlIHRvb2xzOiB7dG9vbHNfc3JjfSIpCiAgICByZXBvcnRbIm5vdGVzIl0uYXBwZW5kKGYi"
        "Y29tcGlsZSBtb2RlbHM6IHttb2RlbHNfc3JjfSIpCiAgICByZWdpc3RyeSA9IFRvb2xSZWdpc3RyeSgpCiAgICBmb3IgaywgdiBp"
        "biB0b29scy5pdGVtcygpOgogICAgICAgIHJlZ2lzdHJ5LnJlZ2lzdGVyKGssIHYpCiAgICBrdyA9IGRpY3Qoc3VibWlzc2lvbl9k"
        "aXI9c3ViX2RpciwgdG9vbF9yZWdpc3RyeT1yZWdpc3RyeSwgbW9kZWxfcmVnaXN0cnk9bW9kZWxzLCBsaW1pdHM9bGltaXRzLAog"
        "ICAgICAgICAgICAgIGdlbmVyYXRpb25fY29uc3RyYWludHM9Z2VuX2NvbnN0cmFpbnRzKQogICAgYXR0ZW1wdHMgPSBbZGljdChr"
        "dywgYWRhcHRlcl9tYW5pZmVzdD1tYW5pZmVzdCwgYWRhcHRlcl9yZXNvbHZlcl9mbj1yZXNvbHZlciwgc2NyaXB0X3RpbWVvdXQ9"
        "MTgwKSwKICAgICAgICAgICAgICAgIGRpY3Qoa3csIGFkYXB0ZXJfcmVzb2x2ZXJfZm49cmVzb2x2ZXIpIGlmIHJlc29sdmVyIGVs"
        "c2UgZGljdChrdyldCiAgICBsYXN0ID0gTm9uZQogICAgZm9yIGEgaW4gYXR0ZW1wdHM6CiAgICAgICAgdHJ5OgogICAgICAgICAg"
        "ICByZXR1cm4gZGVzY3JpYmUoY29tcGlsZV9zdWJtaXNzaW9uKCoqYSkpCiAgICAgICAgZXhjZXB0IFR5cGVFcnJvciBhcyBlOiAg"
        "IyBzaWduYXR1cmUgZHJpZnQgYmV0d2VlbiByZWxlYXNlczogcmV0cnkgd2l0aCB0aGUgc21hbGxlciBrd2FyZyBzZXQKICAgICAg"
        "ICAgICAgbGFzdCA9IGUKICAgICAgICAgICAgcmVwb3J0WyJub3RlcyJdLmFwcGVuZChmImNvbXBpbGVfc3VibWlzc2lvbiBzaWdu"
        "YXR1cmUgbWlzbWF0Y2gsIHJldHJ5aW5nIHdpdGggZmV3ZXIga3dhcmdzOiB7ZX0iKQogICAgcmFpc2UgbGFzdAoKCm9rX2MsIF8g"
        "PSBzdGVwKCJjb21waWxlX3N1Ym1pc3Npb24iLCBkb19jb21waWxlKQpyZXBvcnRbIlZFUkRJQ1RfY29tcGlsZV9zdWJtaXNzaW9u"
        "Il0gPSAiUEFTUyIgaWYgb2tfYyBlbHNlICJGQUlMIgpwcmludChqc29uLmR1bXBzKHJlcG9ydCwgaW5kZW50PTIsIGRlZmF1bHQ9"
        "c3RyKSkK"
    )),
    "replay_http_capture.py": ("5cb66551e71e53d1f1e579dd1e3142fb404126a0602fac3bfef85d22c9a0bafb", (
        "IiIiRkFLRS1NT0RFTCBSRVBMQVkgLSBOT1QgQSBHRU1NQSBURVNUIC0gTk9UIEEgU0NPUkUgVEVTVCAtIE5PVCBUSEUgT0ZGSUNJ"
        "QUwgRVZBTFVBVE9SLgoKQ29tcGlsZXMgYW4gdW5wYWNrZWQgc3VibWlzc2lvbiB3aXRoIHRoZSBJTlNUQUxMRUQgaGFybmVzcyAo"
        "cmVhbCBMaXRlTGxtIG1vZGVsIG9iamVjdHMsIHJlYWwgdGhpbmtpbmctY29uZmlnIHJlc29sdmVyKSwKdGhlbiBkcml2ZXMgT05F"
        "IGJvdW5kZWQgQURLIGNvbnZlcnNhdGlvbiBhZ2FpbnN0IGEgbG9jYWwgc2NyaXB0ZWQgT3BlbkFJLWNvbXBhdGlibGUgSFRUUCBz"
        "ZXJ2ZXIgYW5kIHJlY29yZHMgd2hhdCB0aGUKYWdlbnQgYWN0dWFsbHkgU0VORFMuICBRdWVzdGlvbiBhbnN3ZXJlZDogZG8gdGhl"
        "IHRoaW5raW5nLWRpc2FibGVkIHNldHRpbmdzIGluIHRoZSBzdWJtaXNzaW9uIHJlYWNoIHRoZSByZXF1ZXN0IGJvZHk/CgogICAg"
        "cHl0aG9uIC1JIHJlcGxheV9odHRwX2NhcHR1cmUucHkgPGhhcm5lc3NfZGlyIHwgLT4gPHN1Ym1pc3Npb25fZGlyPiBbLS1vdXQg"
        "cmVzdWx0Lmpzb25dCgpTY3JpcHRlZCBiZWhhdmlvdXIgKHBlciBkaXN0aW5jdCBhZ2VudCBzeXN0ZW0gcHJvbXB0KTogY2FsbCBg"
        "Y29kZV9hbmFseXplcmAgb25jZSBpZiBpdCBpcyBvZmZlcmVkLCBjYWxsIGBnZXRfc3RhdHVzYApvbmNlLCB0aGVuIChyb290IG9u"
        "bHkpIGNhbGwgYHN1Ym1pdF9wYXRjaGAgb25jZSB0aHJvdWdoIGEgRkFLRSBzYW5kYm94IG9iamVjdCwgdGhlbiBhbnN3ZXIgImRv"
        "bmUgKHNjcmlwdGVkKSIuClRoZSBmYWtlIHNhbmRib3ggcmV0dXJucyBhIGNhbm5lZCBkaWZmLCBzbyB0aGUgc3VibWl0X3BhdGNo"
        "IHJlc3VsdCBwcm92ZXMgdG9vbCBwbHVtYmluZyBvbmx5LCBuZXZlciByZWFsIHBhdGNoIGNhcHR1cmUuCgpDaGVja3MgcmVjb3Jk"
        "ZWQ6IGFnZW50IGxvYWRzLCBkZWNsYXJlZCB0b29scyByZXNvbHZlLCB0b29sIGFyZ3VtZW50cyBwYXJzZSwgaXNzdWUgdGV4dCBy"
        "ZWFjaGVzIHRoZSBzeXN0ZW0gcHJvbXB0LAp0b29sIHJlc3VsdHMgZmxvdyBiYWNrLCBubyB1bmRlY2xhcmVkIHRvb2wgaXMgb2Zm"
        "ZXJlZC9jYWxsZWQsIHN1Ym1pdF9wYXRjaCBzZXRzIHBhdGNoX3N1Ym1pdHRlZCwgbm8gcmF3IHt0ZW1wbGF0ZX0KYnJhY2VzIG9y"
        "IGluY2x1ZGUgZXJyb3JzLiAgRXhpdCBjb2RlIGlzIGFsd2F5cyAwOyByZWFkIFZFUkRJQ1QgaW4gdGhlIEpTT04uCiIiIgpmcm9t"
        "IF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGFzeW5jaW8KaW1wb3J0IGpzb24K"
        "aW1wb3J0IG9zCmltcG9ydCBzeXMKaW1wb3J0IHRocmVhZGluZwppbXBvcnQgdGltZQppbXBvcnQgdHJhY2ViYWNrCmZyb20gaHR0"
        "cC5zZXJ2ZXIgaW1wb3J0IEJhc2VIVFRQUmVxdWVzdEhhbmRsZXIsIFRocmVhZGluZ0hUVFBTZXJ2ZXIKZnJvbSBwYXRobGliIGlt"
        "cG9ydCBQYXRoCmZyb20gdHlwZXMgaW1wb3J0IFNpbXBsZU5hbWVzcGFjZQoKZm9yIF9rIGluICgiTk9fUFJPWFkiLCAibm9fcHJv"
        "eHkiKToKICAgIG9zLmVudmlyb25bX2tdID0gIiwiLmpvaW4oZmlsdGVyKE5vbmUsIFtvcy5lbnZpcm9uLmdldChfaywgIiIpLCAi"
        "MTI3LjAuMC4xIiwgImxvY2FsaG9zdCJdKSkKb3MuZW52aXJvbi5zZXRkZWZhdWx0KCJMSVRFTExNX0xPQ0FMX01PREVMX0NPU1Rf"
        "TUFQIiwgIlRydWUiKQpNT0RFTCA9ICJnZW1tYS00LTMxYi1pdC1xYXQtdzRhMTYtY3QiClNFTlRJTkVMID0gIklTU1VFLVNFTlRJ"
        "TkVMLTdmM2E6IHBhcnNlX3JlcXVlc3QoKSBkcm9wcyB0aGUgdHJhaWxpbmcgc2xhc2giCgpDQVBUVVJFRDogbGlzdFtkaWN0XSA9"
        "IFtdClNUQVRFOiBkaWN0ID0geyJjYWxsc19ieV9hZ2VudCI6IHt9LCAibiI6IDB9CkxPQ0sgPSB0aHJlYWRpbmcuTG9jaygpCgoK"
        "ZGVmIGZpbmRfdGhpbmtfa2V5cyhvLCBwYXRoPSIiKToKICAgIG91dCA9IHt9CiAgICBpZiBpc2luc3RhbmNlKG8sIGRpY3QpOgog"
        "ICAgICAgIGZvciBrLCB2IGluIG8uaXRlbXMoKToKICAgICAgICAgICAgcCA9IGYie3BhdGh9LntrfSIgaWYgcGF0aCBlbHNlIHN0"
        "cihrKQogICAgICAgICAgICBpZiBhbnkocyBpbiBzdHIoaykubG93ZXIoKSBmb3IgcyBpbiAoInRoaW5rIiwgInJlYXNvbiIpKToK"
        "ICAgICAgICAgICAgICAgIG91dFtwXSA9IHYKICAgICAgICAgICAgb3V0LnVwZGF0ZShmaW5kX3RoaW5rX2tleXModiwgcCkpCiAg"
        "ICByZXR1cm4gb3V0CgoKY2xhc3MgSGFuZGxlcihCYXNlSFRUUFJlcXVlc3RIYW5kbGVyKToKICAgIGRlZiBsb2dfbWVzc2FnZShz"
        "ZWxmLCAqYSk6ICAjIHNpbGVuY2UKICAgICAgICBwYXNzCgogICAgZGVmIGRvX1BPU1Qoc2VsZik6ICAjIG5vcWE6IE44MDIKICAg"
        "ICAgICBib2R5ID0ganNvbi5sb2FkcyhzZWxmLnJmaWxlLnJlYWQoaW50KHNlbGYuaGVhZGVycy5nZXQoImNvbnRlbnQtbGVuZ3Ro"
        "IiwgMCkpKSBvciBiInt9IikKICAgICAgICBtc2dzID0gYm9keS5nZXQoIm1lc3NhZ2VzIiwgW10pCiAgICAgICAgc3lzdGVtID0g"
        "IlxuIi5qb2luKHN0cihtLmdldCgiY29udGVudCIpKSBmb3IgbSBpbiBtc2dzIGlmIG0uZ2V0KCJyb2xlIikgPT0gInN5c3RlbSIp"
        "CiAgICAgICAgdG9vbHMgPSB7dFsiZnVuY3Rpb24iXVsibmFtZSJdOiB0WyJmdW5jdGlvbiJdIGZvciB0IGluIGJvZHkuZ2V0KCJ0"
        "b29scyIsIFtdKSBpZiB0LmdldCgidHlwZSIpID09ICJmdW5jdGlvbiJ9CiAgICAgICAgd2l0aCBMT0NLOgogICAgICAgICAgICBT"
        "VEFURVsibiJdICs9IDEKICAgICAgICAgICAgbiA9IFNUQVRFWyJuIl0KICAgICAgICAgICAga2V5ID0gaGFzaChzeXN0ZW0pCiAg"
        "ICAgICAgICAgIHNlZW4gPSBTVEFURVsiY2FsbHNfYnlfYWdlbnQiXS5zZXRkZWZhdWx0KGtleSwgc2V0KCkpCiAgICAgICAgICAg"
        "IENBUFRVUkVELmFwcGVuZCh7CiAgICAgICAgICAgICAgICAibiI6IG4sICJwYXRoIjogc2VsZi5wYXRoLCAibW9kZWwiOiBib2R5"
        "LmdldCgibW9kZWwiKSwgInN0cmVhbSI6IGJvb2woYm9keS5nZXQoInN0cmVhbSIpKSwKICAgICAgICAgICAgICAgICJ0b3BfbGV2"
        "ZWxfa2V5cyI6IHNvcnRlZChib2R5LmtleXMoKSksCiAgICAgICAgICAgICAgICAidGhpbmtpbmdfcmVsYXRlZF9maWVsZHMiOiBm"
        "aW5kX3RoaW5rX2tleXMoYm9keSksCiAgICAgICAgICAgICAgICAidG9vbHNfb2ZmZXJlZCI6IHNvcnRlZCh0b29scyksICJuX21l"
        "c3NhZ2VzIjogbGVuKG1zZ3MpLAogICAgICAgICAgICAgICAgImlzc3VlX2luX3N5c3RlbV9wcm9tcHQiOiAiSVNTVUUtU0VOVElO"
        "RUwiIGluIHN5c3RlbSwKICAgICAgICAgICAgICAgICJpc3N1ZV9pbl9hbnlfbWVzc2FnZSI6ICJJU1NVRS1TRU5USU5FTCIgaW4g"
        "anNvbi5kdW1wcyhtc2dzKSwKICAgICAgICAgICAgICAgICJyYXdfdGVtcGxhdGVfYnJhY2VzX2xlZnQiOiAie3Byb2JsZW1fZGVz"
        "Y3JpcHRpb24iIGluIHN5c3RlbSwKICAgICAgICAgICAgICAgICJzeXN0ZW1fcHJvbXB0X2hlYWQiOiBzeXN0ZW1bOjgwXSwKICAg"
        "ICAgICAgICAgfSkKICAgICAgICAgICAgY2FsbCA9IE5vbmUKICAgICAgICAgICAgaWYgbiA+IDI0OgogICAgICAgICAgICAgICAg"
        "c2VsZi5zZW5kX3Jlc3BvbnNlKDUwMCk7IHNlbGYuZW5kX2hlYWRlcnMoKTsgcmV0dXJuCiAgICAgICAgICAgIGlmICJjb2RlX2Fu"
        "YWx5emVyIiBpbiB0b29scyBhbmQgImNvZGVfYW5hbHl6ZXIiIG5vdCBpbiBzZWVuOgogICAgICAgICAgICAgICAgcHJvcHMgPSBs"
        "aXN0KCh0b29sc1siY29kZV9hbmFseXplciJdLmdldCgicGFyYW1ldGVycyIpIG9yIHt9KS5nZXQoInByb3BlcnRpZXMiLCB7fSkp"
        "CiAgICAgICAgICAgICAgICBjYWxsID0gKCJjb2RlX2FuYWx5emVyIiwgeyhwcm9wc1swXSBpZiBwcm9wcyBlbHNlICJyZXF1ZXN0"
        "Iik6ICJwYXJzZV9yZXF1ZXN0IHRyYWlsaW5nIHNsYXNoIn0pCiAgICAgICAgICAgIGVsaWYgImdldF9zdGF0dXMiIGluIHRvb2xz"
        "IGFuZCAiZ2V0X3N0YXR1cyIgbm90IGluIHNlZW46CiAgICAgICAgICAgICAgICBjYWxsID0gKCJnZXRfc3RhdHVzIiwge30pCiAg"
        "ICAgICAgICAgIGVsaWYgInN1Ym1pdF9wYXRjaCIgaW4gdG9vbHMgYW5kICJzdWJtaXRfcGF0Y2giIG5vdCBpbiBzZWVuOgogICAg"
        "ICAgICAgICAgICAgY2FsbCA9ICgic3VibWl0X3BhdGNoIiwge30pCiAgICAgICAgICAgIGlmIGNhbGw6CiAgICAgICAgICAgICAg"
        "ICBzZWVuLmFkZChjYWxsWzBdKQogICAgICAgIGlmIGNhbGw6CiAgICAgICAgICAgIG1lc3NhZ2UgPSB7InJvbGUiOiAiYXNzaXN0"
        "YW50IiwgImNvbnRlbnQiOiBOb25lLCAidG9vbF9jYWxscyI6IFsKICAgICAgICAgICAgICAgIHsiaWQiOiBmImNhbGxfe259Iiwg"
        "InR5cGUiOiAiZnVuY3Rpb24iLCAiZnVuY3Rpb24iOiB7Im5hbWUiOiBjYWxsWzBdLCAiYXJndW1lbnRzIjoganNvbi5kdW1wcyhj"
        "YWxsWzFdKX19XX0KICAgICAgICAgICAgZmluaXNoID0gInRvb2xfY2FsbHMiCiAgICAgICAgZWxzZToKICAgICAgICAgICAgbWVz"
        "c2FnZSA9IHsicm9sZSI6ICJhc3Npc3RhbnQiLAogICAgICAgICAgICAgICAgICAgICAgICJjb250ZW50IjogIlRBUkdFVDogcGFy"
        "c2VfcmVxdWVzdFxuUk9PVCBDQVVTRTogeFxuRklYIFBMQU46IHlcbkNPTkZJREVOQ0U6IGhpZ2giIGlmICJJU1NVRSBUTyBJTlZF"
        "U1RJR0FURSIgaW4gc3lzdGVtCiAgICAgICAgICAgICAgICAgICAgICAgYW5kICJjb2RlX2FuYWx5emVyIiBub3QgaW4gdG9vbHMg"
        "ZWxzZSAiZG9uZSAoc2NyaXB0ZWQpIn0KICAgICAgICAgICAgZmluaXNoID0gInN0b3AiCiAgICAgICAgdXNhZ2UgPSB7InByb21w"
        "dF90b2tlbnMiOiAxMCwgImNvbXBsZXRpb25fdG9rZW5zIjogNSwgInRvdGFsX3Rva2VucyI6IDE1fQogICAgICAgIGlmIGJvZHku"
        "Z2V0KCJzdHJlYW0iKToKICAgICAgICAgICAgc2VsZi5zZW5kX3Jlc3BvbnNlKDIwMCk7IHNlbGYuc2VuZF9oZWFkZXIoImNvbnRl"
        "bnQtdHlwZSIsICJ0ZXh0L2V2ZW50LXN0cmVhbSIpOyBzZWxmLmVuZF9oZWFkZXJzKCkKICAgICAgICAgICAgZGVsdGEgPSBkaWN0"
        "KG1lc3NhZ2UpOyBkZWx0YS5wb3AoInJvbGUiLCBOb25lKQogICAgICAgICAgICBpZiAidG9vbF9jYWxscyIgaW4gZGVsdGE6CiAg"
        "ICAgICAgICAgICAgICBkZWx0YVsidG9vbF9jYWxscyJdID0gW2RpY3QodGMsIGluZGV4PTApIGZvciB0YyBpbiBkZWx0YVsidG9v"
        "bF9jYWxscyJdXQogICAgICAgICAgICBjaHVua3MgPSBbeyJpZCI6ICJjIiwgIm9iamVjdCI6ICJjaGF0LmNvbXBsZXRpb24uY2h1"
        "bmsiLCAiY3JlYXRlZCI6IGludCh0aW1lLnRpbWUoKSksICJtb2RlbCI6IE1PREVMLAogICAgICAgICAgICAgICAgICAgICAgICJj"
        "aG9pY2VzIjogW3siaW5kZXgiOiAwLCAiZGVsdGEiOiB7InJvbGUiOiAiYXNzaXN0YW50IiwgKipkZWx0YX0sICJmaW5pc2hfcmVh"
        "c29uIjogTm9uZX1dfSwKICAgICAgICAgICAgICAgICAgICAgIHsiaWQiOiAiYyIsICJvYmplY3QiOiAiY2hhdC5jb21wbGV0aW9u"
        "LmNodW5rIiwgImNyZWF0ZWQiOiBpbnQodGltZS50aW1lKCkpLCAibW9kZWwiOiBNT0RFTCwKICAgICAgICAgICAgICAgICAgICAg"
        "ICAiY2hvaWNlcyI6IFt7ImluZGV4IjogMCwgImRlbHRhIjoge30sICJmaW5pc2hfcmVhc29uIjogZmluaXNofV0sICJ1c2FnZSI6"
        "IHVzYWdlfV0KICAgICAgICAgICAgZm9yIGMgaW4gY2h1bmtzOgogICAgICAgICAgICAgICAgc2VsZi53ZmlsZS53cml0ZShmImRh"
        "dGE6IHtqc29uLmR1bXBzKGMpfVxuXG4iLmVuY29kZSgpKQogICAgICAgICAgICBzZWxmLndmaWxlLndyaXRlKGIiZGF0YTogW0RP"
        "TkVdXG5cbiIpCiAgICAgICAgICAgIHJldHVybgogICAgICAgIHBheWxvYWQgPSBqc29uLmR1bXBzKHsiaWQiOiAiYyIsICJvYmpl"
        "Y3QiOiAiY2hhdC5jb21wbGV0aW9uIiwgImNyZWF0ZWQiOiBpbnQodGltZS50aW1lKCkpLCAibW9kZWwiOiBNT0RFTCwKICAgICAg"
        "ICAgICAgICAgICAgICAgICAgICAgICAgImNob2ljZXMiOiBbeyJpbmRleCI6IDAsICJtZXNzYWdlIjogbWVzc2FnZSwgImZpbmlz"
        "aF9yZWFzb24iOiBmaW5pc2h9XSwgInVzYWdlIjogdXNhZ2V9KS5lbmNvZGUoKQogICAgICAgIHNlbGYuc2VuZF9yZXNwb25zZSgy"
        "MDApOyBzZWxmLnNlbmRfaGVhZGVyKCJjb250ZW50LXR5cGUiLCAiYXBwbGljYXRpb24vanNvbiIpCiAgICAgICAgc2VsZi5zZW5k"
        "X2hlYWRlcigiY29udGVudC1sZW5ndGgiLCBzdHIobGVuKHBheWxvYWQpKSk7IHNlbGYuZW5kX2hlYWRlcnMoKTsgc2VsZi53Zmls"
        "ZS53cml0ZShwYXlsb2FkKQoKCmNsYXNzIEZha2VTYW5kYm94OgogICAgIiIiU3RhbmRzIGluIGZvciB0aGUgRG9ja2VyL3N1YnBy"
        "b2Nlc3Mgc2FuZGJveC4gUmV0dXJucyBhIGNhbm5lZCBkaWZmLiBOT1QgYSByZWFsIHNhbmRib3guIiIiCiAgICBkZWYgZXhlYyhz"
        "ZWxmLCBjb250YWluZXJfaWQsIGNtZCwgKmEsICoqayk6CiAgICAgICAgb3V0ID0gImRpZmYgLS1naXQgYS94LnB5IGIveC5weVxu"
        "LS0tIGEveC5weVxuKysrIGIveC5weVxuQEAgLTEgKzEgQEBcbi1hXG4rYlxuIiBpZiAiZ2l0IGRpZmYiIGluIGNtZCBlbHNlICIi"
        "CiAgICAgICAgcmV0dXJuIFNpbXBsZU5hbWVzcGFjZShleGl0X2NvZGU9MCwgc3Rkb3V0PW91dCwgc3RkZXJyPSIiKQoKICAgIGRl"
        "ZiBfX2dldGF0dHJfXyhzZWxmLCBuYW1lKTogICMgYW55IG90aGVyIHNhbmRib3ggY2FsbDogaGFybWxlc3Mgbm8tb3AKICAgICAg"
        "ICByZXR1cm4gbGFtYmRhICphLCAqKms6IFNpbXBsZU5hbWVzcGFjZShleGl0X2NvZGU9MCwgc3Rkb3V0PSIiLCBzdGRlcnI9IiIp"
        "CgoKZGVmIGFnZW50X3N1bW1hcnkoYSk6CiAgICBhYiA9IGdldGF0dHIoZ2V0YXR0cihhLCAibW9kZWwiLCBOb25lKSwgIl9hZGRp"
        "dGlvbmFsX2FyZ3MiLCBOb25lKQogICAgcmV0dXJuIHsibmFtZSI6IGEubmFtZSwgIm1vZGVsX2NsYXNzIjogdHlwZShnZXRhdHRy"
        "KGEsICJtb2RlbCIsIE5vbmUpKS5fX25hbWVfXywKICAgICAgICAgICAgImV4dHJhX2JvZHlfb25fbW9kZWwiOiAoYWIgb3Ige30p"
        "LmdldCgiZXh0cmFfYm9keSIpIGlmIGlzaW5zdGFuY2UoYWIsIGRpY3QpIGVsc2UgTm9uZSwKICAgICAgICAgICAgInRvb2xzIjog"
        "W2dldGF0dHIodCwgIm5hbWUiLCBnZXRhdHRyKHQsICJfX25hbWVfXyIsIHR5cGUodCkuX19uYW1lX18pKSBmb3IgdCBpbiBnZXRh"
        "dHRyKGEsICJ0b29scyIsIFtdKV0sCiAgICAgICAgICAgICJzdWJfYWdlbnRzIjogW2FnZW50X3N1bW1hcnkocykgZm9yIHMgaW4g"
        "Z2V0YXR0cihhLCAic3ViX2FnZW50cyIsIFtdKV0sCiAgICAgICAgICAgICJhZ2VudF90b29scyI6IFthZ2VudF9zdW1tYXJ5KHQu"
        "YWdlbnQpIGZvciB0IGluIGdldGF0dHIoYSwgInRvb2xzIiwgW10pIGlmIGhhc2F0dHIodCwgImFnZW50IildfQoKCmFzeW5jIGRl"
        "ZiBydW4odXJsOiBzdHIsIFNVQjogUGF0aCkgLT4gZGljdDoKICAgIGZyb20gZ29vZ2xlLmFkay5hcHBzIGltcG9ydCBBcHAKICAg"
        "IGZyb20gZ29vZ2xlLmFkay5tb2RlbHMubGl0ZV9sbG0gaW1wb3J0IExpdGVMbG0KICAgIGZyb20gZ29vZ2xlLmFkay5ydW5uZXJz"
        "IGltcG9ydCBSdW5uZXIKICAgIGZyb20gZ29vZ2xlLmFkay5zZXNzaW9ucyBpbXBvcnQgSW5NZW1vcnlTZXNzaW9uU2VydmljZQog"
        "ICAgZnJvbSBnb29nbGUuZ2VuYWkgaW1wb3J0IHR5cGVzCgogICAgZnJvbSBhZGtfc3VibWlzc2lvbiBpbXBvcnQgTW9kZWxSZWdp"
        "c3RyeSwgVG9vbFJlZ2lzdHJ5LCBjb21waWxlX3N1Ym1pc3Npb24KICAgIGZyb20gc3dlZ2VtbWEuY29uZmlnIGltcG9ydCBidWls"
        "ZF9zdWJtaXNzaW9uX2xpbWl0cwogICAgZnJvbSBzd2VnZW1tYS5jb250ZXh0IGltcG9ydCBTd2VnZW1tYUNvbnRleHQKCiAgICBv"
        "dXQ6IGRpY3QgPSB7InN1Ym1pc3Npb25fZGlyIjogc3RyKFNVQil9CiAgICBsaW1pdHMsIGdjID0gYnVpbGRfc3VibWlzc2lvbl9s"
        "aW1pdHMoKQogICAgbW9kZWxzID0gTW9kZWxSZWdpc3RyeSgpCiAgICBtb2RlbHMucmVnaXN0ZXIoTU9ERUwsIExpdGVMbG0obW9k"
        "ZWw9ZiJvcGVuYWkve01PREVMfSIsIGFwaV9iYXNlPXVybCwgYXBpX2tleT0iRU1QVFkiLCBudW1fcmV0cmllcz0wKSkKICAgIGN0"
        "eCA9IFN3ZWdlbW1hQ29udGV4dChkb2NrZXJfbWFuYWdlcj1GYWtlU2FuZGJveCgpLCBjb250YWluZXJfaWQ9ImZha2UiLCBwcm9i"
        "bGVtX3N0YXRlbWVudD1TRU5USU5FTCwgdGFza19pZD0idCIsIHJlcG89InIiLAogICAgICAgICAgICAgICAgICAgICAgICAgIG1h"
        "eF90b29sX2NhbGxzPTQwLCBtYXhfdGltZV9taW51dGVzPTQuNSkKICAgIHRyID0gVG9vbFJlZ2lzdHJ5KCkKICAgIGZvciBrLCB2"
        "IGluIGN0eC5jcmVhdGVfdG9vbHMoKS5pdGVtcygpOgogICAgICAgIHRyLnJlZ2lzdGVyKGssIHYpCiAgICBhZ2VudCA9IGNvbXBp"
        "bGVfc3VibWlzc2lvbihzdWJtaXNzaW9uX2Rpcj1TVUIsIHRvb2xfcmVnaXN0cnk9dHIsIG1vZGVsX3JlZ2lzdHJ5PW1vZGVscywg"
        "bGltaXRzPWxpbWl0cywKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGdlbmVyYXRpb25fY29uc3RyYWludHM9Z2MsIHNj"
        "cmlwdF90aW1lb3V0PTE4MCkKICAgIG91dFsiY29tcGlsZWRfYWdlbnRzIl0gPSBhZ2VudF9zdW1tYXJ5KGFnZW50KQogICAgc3Zj"
        "ID0gSW5NZW1vcnlTZXNzaW9uU2VydmljZSgpCiAgICBzZXNzID0gYXdhaXQgc3ZjLmNyZWF0ZV9zZXNzaW9uKGFwcF9uYW1lPSJy"
        "ZXBsYXkiLCB1c2VyX2lkPSJ1Iiwgc3RhdGU9eyJwcm9ibGVtX2Rlc2NyaXB0aW9uIjogU0VOVElORUx9KQogICAgcnVubmVyID0g"
        "UnVubmVyKGFwcD1BcHAobmFtZT0icmVwbGF5Iiwgcm9vdF9hZ2VudD1hZ2VudCksIHNlc3Npb25fc2VydmljZT1zdmMpCiAgICBu"
        "dWRnZSA9ICgiUGxlYXNlIGNvbnRpbnVlIHlvdXIgd29yayB1c2luZyB0aGUgYXZhaWxhYmxlIHRvb2xzLCBvciBjYWxsIHN1Ym1p"
        "dF9wYXRjaCB3aGVuICIKICAgICAgICAgICAgICJ5b3UgaGF2ZSBjb21wbGV0ZWQgYW5kIHZlcmlmaWVkIHlvdXIgY2hhbmdlcy4i"
        "KQogICAgbXNnLCBldmVudHMgPSAiIyMgUHJvYmxlbSBTdGF0ZW1lbnRcbiIgKyBTRU5USU5FTCArICJcblxuUmVzb2x2ZSBpdC4i"
        "LCBbXQogICAgZm9yIGludiBpbiByYW5nZSgxLCA1KToKICAgICAgICBldnMgPSBbXQogICAgICAgIGFzeW5jIGZvciBldiBpbiBy"
        "dW5uZXIucnVuX2FzeW5jKHVzZXJfaWQ9InUiLCBzZXNzaW9uX2lkPXNlc3MuaWQsCiAgICAgICAgICAgICAgICAgICAgICAgICAg"
        "ICAgICAgICAgICAgICAgbmV3X21lc3NhZ2U9dHlwZXMuQ29udGVudChyb2xlPSJ1c2VyIiwgcGFydHM9W3R5cGVzLlBhcnQodGV4"
        "dD1tc2cpXSkpOgogICAgICAgICAgICBraW5kcyA9IFtdCiAgICAgICAgICAgIGZvciBwIGluIChldi5jb250ZW50LnBhcnRzIGlm"
        "IGV2LmNvbnRlbnQgYW5kIGV2LmNvbnRlbnQucGFydHMgZWxzZSBbXSk6CiAgICAgICAgICAgICAgICBpZiBwLmZ1bmN0aW9uX2Nh"
        "bGw6CiAgICAgICAgICAgICAgICAgICAga2luZHMuYXBwZW5kKGYiY2FsbDp7cC5mdW5jdGlvbl9jYWxsLm5hbWV9IikKICAgICAg"
        "ICAgICAgICAgIGlmIHAuZnVuY3Rpb25fcmVzcG9uc2U6CiAgICAgICAgICAgICAgICAgICAga2luZHMuYXBwZW5kKGYicmVzcDp7"
        "cC5mdW5jdGlvbl9yZXNwb25zZS5uYW1lfSIpCiAgICAgICAgICAgICAgICBpZiBwLnRleHQ6CiAgICAgICAgICAgICAgICAgICAg"
        "a2luZHMuYXBwZW5kKCJ0ZXh0OiIgKyBwLnRleHRbOjMwXS5yZXBsYWNlKCJcbiIsICIgIikpCiAgICAgICAgICAgIGV2cy5hcHBl"
        "bmQoeyJhdXRob3IiOiBldi5hdXRob3IsICJwYXJ0cyI6IGtpbmRzfSkKICAgICAgICBldmVudHMuYXBwZW5kKHsiaW52b2NhdGlv"
        "biI6IGludiwgImV2ZW50cyI6IGV2c30pCiAgICAgICAgaWYgYW55KGsuc3RhcnRzd2l0aCgidGV4dDpkb25lIikgZm9yIGUgaW4g"
        "ZXZzIGZvciBrIGluIGVbInBhcnRzIl0pOgogICAgICAgICAgICBicmVhawogICAgICAgIG1zZyA9IG51ZGdlCiAgICBvdXRbImV2"
        "ZW50cyJdID0gZXZlbnRzCiAgICBvdXRbImN0eF9hZnRlciJdID0geyJwYXRjaF9zdWJtaXR0ZWQiOiBib29sKGdldGF0dHIoY3R4"
        "LCAicGF0Y2hfc3VibWl0dGVkIiwgRmFsc2UpKSwKICAgICAgICAgICAgICAgICAgICAgICAgInN1Ym1pdHRlZF9wYXRjaF9jaGFy"
        "cyI6IGxlbihnZXRhdHRyKGN0eCwgInN1Ym1pdHRlZF9wYXRjaCIsICIiKSBvciAiIiksCiAgICAgICAgICAgICAgICAgICAgICAg"
        "ICJ0b29sX2NhbGxzX3VzZWQiOiBjdHgudG9vbF9jYWxsc191c2VkfQogICAgcmV0dXJuIG91dAoKCmRlZiBzdGFydF9zZXJ2ZXIo"
        "KToKICAgIHNydiA9IFRocmVhZGluZ0hUVFBTZXJ2ZXIoKCIxMjcuMC4wLjEiLCAwKSwgSGFuZGxlcikKICAgIHRocmVhZGluZy5U"
        "aHJlYWQodGFyZ2V0PXNydi5zZXJ2ZV9mb3JldmVyLCBkYWVtb249VHJ1ZSkuc3RhcnQoKQogICAgcmV0dXJuIHNydiwgZiJodHRw"
        "Oi8vMTI3LjAuMC4xOntzcnYuc2VydmVyX2FkZHJlc3NbMV19L3YxIgoKCmRlZiBtYWluKCk6CiAgICBhcCA9IGFyZ3BhcnNlLkFy"
        "Z3VtZW50UGFyc2VyKCkKICAgIGFwLmFkZF9hcmd1bWVudCgiaGFybmVzc19kaXIiKQogICAgYXAuYWRkX2FyZ3VtZW50KCJzdWJt"
        "aXNzaW9uX2RpciIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tb3V0IiwgZGVmYXVsdD0iIikKICAgIGFyZ3MgPSBhcC5wYXJzZV9h"
        "cmdzKCkKICAgIGlmIGFyZ3MuaGFybmVzc19kaXIgIT0gIi0iOgogICAgICAgIHN5cy5wYXRoLmluc2VydCgwLCBhcmdzLmhhcm5l"
        "c3NfZGlyKQogICAgU1VCID0gUGF0aChhcmdzLnN1Ym1pc3Npb25fZGlyKQogICAgQ0FQVFVSRUQuY2xlYXIoKTsgU1RBVEUudXBk"
        "YXRlKGNhbGxzX2J5X2FnZW50PXt9LCBuPTApCiAgICBzcnYsIHVybCA9IHN0YXJ0X3NlcnZlcigpCiAgICB0cnk6CiAgICAgICAg"
        "cmVzID0gYXN5bmNpby5ydW4ocnVuKHVybCwgU1VCKSkKICAgICAgICByZXNbIlZFUkRJQ1QiXSA9ICJSQU5fVE9fQ09NUExFVElP"
        "TiIKICAgIGV4Y2VwdCBCYXNlRXhjZXB0aW9uIGFzIGU6ICAjIG5vcWE6IEJMRTAwMQogICAgICAgIHJlcyA9IHsiVkVSRElDVCI6"
        "ICJFWENFUFRJT04iLCAiZXhjX3R5cGUiOiBmInt0eXBlKGUpLl9fbW9kdWxlX199Lnt0eXBlKGUpLl9fbmFtZV9ffSIsICJtZXNz"
        "YWdlIjogc3RyKGUpLAogICAgICAgICAgICAgICAidHJhY2ViYWNrX3RhaWwiOiB0cmFjZWJhY2suZm9ybWF0X2V4YygpLnN0cmlw"
        "KCkuc3BsaXRsaW5lcygpWy0xNDpdfQogICAgc3J2LnNodXRkb3duKCkKICAgIHJlc1sibGFiZWwiXSA9ICJGQUtFLU1PREVMIFJF"
        "UExBWSAtIE5PVCBBIEdFTU1BIFRFU1QgLSBOT1QgQSBTQ09SRSBURVNUIgogICAgcmVzWyJodHRwX3JlcXVlc3RzX2NhcHR1cmVk"
        "Il0gPSBsaXN0KENBUFRVUkVEKQogICAgYWxsb3dlZCA9IHsicnVuX2NvbW1hbmQiLCAicmVhZF9maWxlIiwgImVkaXRfZmlsZSIs"
        "ICJ3cml0ZV9maWxlIiwgImdldF9zdGF0dXMiLCAic3VibWl0X3BhdGNoIiwKICAgICAgICAgICAgICAgInNlYXJjaF9zaW1pbGFy"
        "X2NvZGUiLCAiZ2V0X2NvZGVfbmVpZ2hib3JzIiwgImdldF9jb2RlX3N1YmdyYXBoIiwgImNvZGVfYW5hbHl6ZXIifQogICAgb2Zm"
        "ZXJlZCA9IHt0IGZvciBjIGluIENBUFRVUkVEIGZvciB0IGluIGNbInRvb2xzX29mZmVyZWQiXX0KICAgIHJlc1siY2hlY2tzIl0g"
        "PSB7CiAgICAgICAgInVuZGVjbGFyZWRfdG9vbHNfb2ZmZXJlZCI6IHNvcnRlZChvZmZlcmVkIC0gYWxsb3dlZCksCiAgICAgICAg"
        "Imlzc3VlX3JlYWNoZWRfYV9zeXN0ZW1fcHJvbXB0IjogYW55KGNbImlzc3VlX2luX3N5c3RlbV9wcm9tcHQiXSBmb3IgYyBpbiBD"
        "QVBUVVJFRCksCiAgICAgICAgImlzc3VlX3JlYWNoZWRfdGhlX3Jvb3RfY29udmVyc2F0aW9uIjogYW55KGNbImlzc3VlX2luX2Fu"
        "eV9tZXNzYWdlIl0gZm9yIGMgaW4gQ0FQVFVSRUQpLAogICAgICAgICJyYXdfdGVtcGxhdGVfYnJhY2VzX2xlZnQiOiBhbnkoY1si"
        "cmF3X3RlbXBsYXRlX2JyYWNlc19sZWZ0Il0gZm9yIGMgaW4gQ0FQVFVSRUQpLAogICAgICAgICJwYXRjaF9zdWJtaXR0ZWRfdmlh"
        "X2Zha2Vfc2FuZGJveCI6IHJlcy5nZXQoImN0eF9hZnRlciIsIHt9KS5nZXQoInBhdGNoX3N1Ym1pdHRlZCIpLAogICAgICAgICJ0"
        "aGlua2luZ19maWVsZHNfc2Vlbl9pbl9hbnlfcmVxdWVzdCI6IHNvcnRlZCh7ayBmb3IgYyBpbiBDQVBUVVJFRCBmb3IgayBpbiBj"
        "WyJ0aGlua2luZ19yZWxhdGVkX2ZpZWxkcyJdfSksCiAgICB9CiAgICB0ZXh0ID0ganNvbi5kdW1wcyhyZXMsIGluZGVudD0yLCBk"
        "ZWZhdWx0PXN0cikKICAgIGlmIGFyZ3Mub3V0OgogICAgICAgIFBhdGgoYXJncy5vdXQpLndyaXRlX3RleHQodGV4dCwgZW5jb2Rp"
        "bmc9InV0Zi04IikKICAgIHByaW50KHRleHQpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo="
    )),
    "candidates/submission-VNEXT.zip": ("292c4ab28c9c516f26f92b71e9352f0e5d87dbfef95b0120845b77f6f62c00da", (
        "UEsDBBQAAAAIACJ8R11asgQb8AAAAJ0BAAAKAAAAYWdlbnQueWFtbFVQW27DIBD85xT0ACSKEvXDl0EbvMGr8uqyluWcvmCipuUD"
        "aZbZmWESRJx03dC6PCOr2O4waY8xgrmZ6+VuSMw3iNlucPk0TpSiVIVXJ5TTpD8oubDOqAvnWKSe614F4ynOSknOoU5Ka6N5Tc2h"
        "iaZ5YITZPijggXAmeaONSfANPYqtArLWA9b1Hhu7gLjl972HtwnJL/fMLx4Cu8VWihSAD8Z/ehPyDGWIgMcktgfueftxOT3Id59l"
        "6qb2oNTzsQsJwv5EPu0Qw2uhflFpou2PTE8Y7bSeUKm2iAzSO07SfYb2n/LGoJUHsQRKfuj+AFBLAwQUAAAACAAifEddhtXOyE4A"
        "AABhAAAAEAAAAGV2YWxfY29uZmlnLnlhbWxLLUvMKU0syczPs+JSUCjJzE3NLy2JL05Nzs9LKbZSMDQyAArnJlbEl+Tn58QnJ+bk"
        "AEVN4IJA9fG5mXmlJakgYT1TmHhpUR5It4EBFwBQSwMEFAAAAAgAInxHXfrLwoZeAAAAagAAAB4AAABjb25maWdzL2FuYWx5emVy"
        "X3NhbXBsaW5nLnlhbWwdylEKgzAMAND/nCInkG44mL1MKBrbUk2KJrDj6/x7H89473wk84MjhuEFpp36n9PncYs4BtjTj9Stu5Fp"
        "YzkjvsP4BStVWpVMs8pacwTEKvPmC5MV9VzsnmvaToYLUEsDBBQAAAAIACJ8R11UM8t9XgAAAGoAAAAVAAAAY29uZmlncy9zYW1w"
        "bGluZy55YW1sHcpRCoMwDADQ/5wiJxA3uoG9TCga21JNiibg8Z37ex/PeO98JPODI47DG0w79YfT5+8WMYywp4vUrbuRaWM5I35f"
        "IYCVKq1KplllrTkCYpV584XJinou9ptr2k6GG1BLAwQUAAAACAAzfEddT6F0HxULAADBFgAAEQAAAHByb21wdHMvc3lzdGVtLm1k"
        "fVhdc9u4FX33r8DkYRN3JPkj8Taj7O6MEsuNp147Yzvdpi8eiARFrCmCC4CWtdMf33MuQElOO32xJRG8uB/nnnuAb65X2hulW6X7"
        "6Fq3cn1QwVVxzZ9Nu7StMX6ibk1wzZNRsTbKhtDjbytfvmxi7VrlTeeCjc5vlI7qaO38Y+h0YSbqG7YIfVEYU6p1bVpV27LEv2hC"
        "DGrtbYz4Vjkv5vouRG/0SlX2WXU6BDW7Ppcnux1eB2WebYi2XWYr+Nw0snyiPu+b70Ny2TzrIqpWr0wYYZ3Hh2g8PpvnwnTRIoK4"
        "6fgQK4JemoCUlMr1sesjnVtpWNsFX5pQeLvgC8ENu6zUk/ELHe1qcnBw+/VqfqduLtT957m6urn5cjBWczzfMJBmo1Z9iKpwbdRI"
        "pFbRuUYVGlH0LYJRG2St1kg4f0PmQr9Y2fjQ6VjUEzVDeM9x7NpmMLfWIdJpFXvfTrDX/gtqfn1+J95HHR4n6hP3sVG5tjAjpQN3"
        "88h4q5udI3hQIUeMyVYbyTUMoCwTdW3wozIlTKQ1Nk7UPWubnbGxhnGjQu18VMG0KDGwALe+cadUjcbCwSDwy1iid0MB1Bvv+mUN"
        "a+/GZ2pl2z7morw73jnJ/NewUKY9aUQjis2fxh9Ocr6ZEWRacGKQ8TJMVVEz7YVbrWAxpJd/+EHMh+g6eNg1zjNo5kiq8di69S4F"
        "l5VCnYHKvg2EyUgeEfaSKW+IFLjYYXd4x+YChCx93ghmVvoxQ5NpnM9ur77J9o/GYHtxvLINUrgBRIr8pnrSjS3ZYawgHQhM6k2C"
        "KTfs43QXFtadHR8fM1oPGwJ4NFf5IJbx9OTsGGVoYUV9BYaXKJ8at/APLo9b9Xo2+ti9Hqka7yBAbZvdupFqXVR+KYUHUTxZ79oV"
        "Kk03XFXR7BSFDZGuYm2NxNDZWxiTwD/bNpVE2s9V/w0BW6nOG8Lng8C1IlPAIJEgvUhclfAV7WiXAvuZ4HEfB4iYCAipQsgEwg1I"
        "/4a4RC7Vq64xGlGxGQEz82rYn6FpiaclATRTtXSAnrq7n39Rp+hx+XCisOv17Orbv+bqzYmg8vBAOqxwpXnYunFz/WmegKbVyfit"
        "5B2+/dGDqKbi26PZKAvuirayqJWqvFvtcc6bqm8lV0dFQ2LMbGa8J3eCD/aYbXi4cmWPUtdMNRriksAhAhh8TmLVkwy4g+CydFLY"
        "jnwifY13cgbV/ez2b/P7kbq9ublXn2Zf7+YjdXH5T/XlanaNX+dXs/v5+Ujdz+/u70bq0831xeX5HGFPETJo3aEz6EmCTvR9BJXd"
        "PdoucRHgbau9eAdPJZQ9Fk/gRemGfAyEVGRai2tLsknlOUV5xJPbX4WKUwyoFEogDHII1OyagtsUNgJT0hfqDeDm4wO/HAEu8gHM"
        "uEDLqx+PxyfHuYEOxSNAqLI+FY3VVwtDEg/k2GFklITpZbWdaV4QZlZd3CgUcs00sQO1B3GTmINpqgScvSn2HTimuXfBdK90W4B2"
        "X6nx2LZF05fm579Muo2aqH/nRU9pOB7hB3a2Gp8eS3O0kgcxWhpMA5uSO4uAEYbV28EpSdteFKlcya3WgbFWC4fxAQvJWIvXsKEK"
        "dtGQHb/DcY3MNcN8CbDCNBYGoTClwDXblGUdHq/1ZqK+eFOh5pqLg/FPINcnVqYEnVrOTZgFhbgnWQQO3lLEeUI4h/bSxAfpUkzL"
        "pdddnfiNO4Gw8Kb8KOMmJITu1SHFOXRaKxAcRp2k5+7lJNEUVmC6NG+RRXAZWGhdW+R0yAmxIBCQ4Nfkn5TCdcKSBUREOiy2CH8L"
        "hM/PL4FpYA9PZKAsNmmo/xhTvT7kSf0+VU/iIIbhBNeP0lsccS7N4gUrZp7IR4Vhj/w6DKywoiShekkpjnQSQzE36Y4cONY7DwZK"
        "kTHRqfV/ZwBiagPYu9UExilTCBQ4HuKmAfn+nZOw64GZQs2+XGZF1pS7RFA09q03jWbD2hZDMOFyBUqXWSSVYtmGRCasIu8QAUzq"
        "ljH3pWDCJbfLk2D0UgkCZgXX7jYdWqFxS3hbo8QgoZb0sWmLI82/mN3eavAwaHpT4gMWPp0cPZ2qhWfLipaEJGq0Tw6AVEbQwyWC"
        "y4KANkE3gRv3GHx+4FwG/ZltBNwukBpoeDBUW2JHg0CvHUkrUQwUBJk+YYyadyQ5lTgkvCyms5ShZUJD2JHSAkyC9Q+gc2J6yK+w"
        "kcY/E2phkVHqlNM86CgJZMCinTllrJBe31qMv1xpowUB68F0CnBg2mxFBFrXEOB2uSTnCwYDpQCYh2mChqXDCJu9H+yfSNZOL7Ll"
        "OXragpiRum3DUxUUDmrWUp0Rj85vTcoPUhFvxluaHIQsXZN4s3CimxiafjMM/NJWYCvD2bdyELyhMwWmfLGXyw+ZQkTwlDu2o1PM"
        "R1qV5RabFQv7Zm+g126dOpCbySwPnK1LLOUvrCRPWybFKmyGnBV4GwDT0g1ZLmSVyzxhEQ6A5Zbg8IY3YkaoYziG0aRQXuElcykB"
        "aH7FoddB50E6AbxHcdWpJ6upmx+yTlVvUAv1izw7kpc4rn766fX85uK1mkyg428uIF5m7UZEZHaZmA2gp73DJpgBNlELdDXlYqLA"
        "dGaCc3vHFhmAI5nX/IgN2ZXyEXOPn+HG75CmE9BTw3Ec+25SVJjM0T1zjXqDLG3k9MKKkZArVhYTo4STh4MoQaBSAuiIUI/SWmxc"
        "m+KRyHZsdoNUgnZ2gVAwUd0LQyfiSg27ckJra9ejZxsXzHB0ex6mwTtMg3/Mby8vvjGvRHxqoffj0/eHB/pQ3fFA8bxXIdvKWY2L"
        "p0wCj/PjFT6xQB2xcnF5NefxKD98WajTX344gZLg6YBKYqI+9rYpB3kD+y+VCg7vobVdZ6DpQ9Z+SfZ5k6R2CyQtDtW1hrBdS6WU"
        "DLaJurCiBLZip4Ha4fh9lU/7g545m6jbvp0eKPUiHjGVpA//PjzT/fHjYGP8rMZ/QDXFxc/p3Dru4M20AC8xDo5C/zLat8cHxSEO"
        "NHAnBA4OFDtJ+XXt2EfJd3ZblfgtN5jMf2EuoD9LntQLw1wePx4KOe2v1MNZtzZNx4kFmAf+J40LklAvHfOgddQ9eXxS+mieRDG5"
        "GhADQcma6/J3oK2Ne57KNIc2ChT/B+WhuoCh7cTNdDuSt4fcFgnOCj5tx48pl4RUQC3NYbpwoNji4a0P6VCdGwbTrFMFov2NVJlY"
        "/69nw1FdNaZCvwhShqPXGa9XjLSZsCPiClP092/SIjLgGH0e2bxeoehCpjjcp6ljthElZt7T0xBs6HudtO/tHtnnU0nm5q3BD+mq"
        "6qXOySoEKJrzeDaCiAZ9L/po8ve+TfsYnvg3aHyU1y97dnf2MOlpjqCheNnX7xqKTnLtZJuBl3daNJFlzDTPtV00Mo/zEKeBi5RO"
        "le6kCAoCsLRlOrK5vqhHw0yLHthZaBReMJPmQyfCBb/I7U2KZMCvzMnpTrHhUL/EyVo8v7DPCZ9d0y/JSEQzZGZKqZxxpy9uGeT2"
        "6MP2gk8GS+IbvPodMpMaH4Yp45Mj4jTd9ch8kAmb1Gh4WXY5x2FNvQeRlqO9c3KWFc2Uxme96XhsCDbsn5p0jJReQfpjb0km7DMQ"
        "9t3Xj79e3vMm5/8T8Xz26fMDtf78/GEg5Txc2FbjsRDXBGZ4mB5uRwKKUBhp73TxoUG/2n9QdLyl3va8Dcmjexjlcp8nR+r9q8R8"
        "UPwfl3uYz1lipjtH3eCcFpKmmpLi03qikhdnIt2DlDzZH/oyxHHoO7YOXF9h0uK4IS0G92q520E7b++T/gNQSwMEFAAAAAgALHxH"
        "XRVcOGFeBQAAFwoAABMAAABwcm9tcHRzL2FuYWx5emVyLm1kfVZtb9s2EP6uX3Hwl9qDLadZUxQuisFNnMVA3mC7awMMMGiJtghL"
        "pEZScYR1/33PkXISbN0+tGb4cnfPc8/d6cE0JKwkQVaKfGR02VJmcklKP0rn1U54Y2mLf76QdN/6wmhcrY1TOGhJeBofjN27WmQy"
        "pak2uGdJ7KT2dFBlSZXYy/B4q54+Ugt/W6Vz+no1W8xIhNVDSg/Yrxrn6fZuRZXJ1bYdUoaYvCQ4z2UpsRK6xetSDsk2mjwCdEM+"
        "Vtp5AV8494XSuzRJ5svllxmt7mh++9tsuZr/Ol3Nkj9razalrNa5dJlVtVdG//JXklxNFxd0Pb+Zr5bJiKYeESCU9+SNKSmDZQcX"
        "+AtOhoxF08EqH2ExGdYHBBbUaS+fPClHrsK7CdVWgQnRmRQb08DwCZVKS0c1qGLz1C/A/pCczGmk6c10+Ll+M0jpi5MhL2sGDTp9"
        "QQBq/Zqfj6XOw4L60eyHo9lKPA1SAOHnO8Q3JKSF7I73nBQ2K9ZOVaoUdh1yzVnYSR/+WGupdsXGWEeeM7d8uPl8d02305vZkvos"
        "BPkkqhrh9M6NdqaUaYDY4zz0amGdXDMYaXsD+JWPgOggBqkz6SKiH4UQsO0/nf1XLOEcuF62Pr09G4bbQbMG5oO49tocsM1RZp5c"
        "W21MCTnczFZXdxfJ25TuVban09E7XMoKNr21pgqZVM41kYyY1z8aCGzSWVI5QKitkhaSk9ZyRSDTQwJkUUGcljR+cVg3m1JlNL2f"
        "x500OU1pGTCTg0gQKIc8CamhkdXUi6H0aDRSOiubXH76Ka1bSul7d+mRekHt4x62mF4anZ6kdMmVlEsUlGIpdzS92MVRiOH371kp"
        "nAvrH3phkynNtywULiBQ7bOC0XjL5CJpCJBBbq3YVVzcIMC12ui2CiWIEmHW0Ab2KP6YRo4DtdxALB0TP6ev9Mz3X4I/qvj9yehZ"
        "yMKaBoaUh5zfhfj4zUYW4lGx/1oAs2ONiZK2jc4CC8PYYbiwkK3XPHMU/f+lObL8TDJrDI7+Lcg0OUvp2mSi6wLySaFbgjg2EHrU"
        "s9+yc9yLxp9tn0GWq7ZWiJNKAe3irA8oaH0xj6ULzaalR4EkbPALOAA4mNClcJ4FtpeydlSaHQSHDrXFtqjVGLRxMAA3BMM1GgXK"
        "T0k3xnbpwnaQbVwaXOBXL4cAW9XChzVeArmNB6KuoW0RaMbGR1ooqBqeLX7H2bEh4CYXR1g433ZblbD7po6nYtNtOhnkFNet9uIp"
        "hmcN6HOdk1iKoQk7m4270nRjiEt2ITtc7qIa0gsSkYvaH6PPjNmDhWDzarW6/8YGC+/rp/E6K9UxjPUrs+vGvnDCXYyvpLQK4VQq"
        "9IHXEifoVdpORjyGtq9bi9u7MEsF3c6+0hbTrbFhvj03kSGtpotfZyueIGFodqLmS7GGvYHYEEgeu99ido3JdtF55ydZaRyL0Cn0"
        "IShScbdmjkPa+H0Xd18w49lY8P/kD0oHsbMNB2vQRfSsEe9R+CjEZDG7v1usqI92TKcnJ4T5n3PfKwUPydAUj/OOO0fo706rupbY"
        "2NKHWNyDSRKhYkYKX0zCYBsBV4B1xD0OoJPFHT4LzqdflrMJHQpYD0DZcG4Am3s+F0fjXk4j5SwVBSFxtHaPfG+lDaMIySDXZMUg"
        "uZx/o/vr6e2kK2Tu9lkh9A6fGY1jBrvG7tROh4y5ECL2HFrdPwdI0mVkQvFTCLSwOy1lHvrDkd3ogvrMO4x33WrYvYqacgNOCJqs"
        "TFb4iFlOftBn+rgUZhZvdCow4fuon5s4+PlbqZDVIDm/u72cX8xuz0FjgYTSGChy1VRYlOYQFXUo2uRvUEsDBBQAAAAIACJ8R12Z"
        "D5qXfwEAADMCAAAdAAAAc3ViX2FnZW50cy9jb2RlX2FuYWx5emVyLnlhbWxNUU1r3EAMvc+vUG8t2A5LQg8+FJaNUxbCbtl1oTcz"
        "sWVbdD5cjZzU/fWVvZT0JJ40896TXrAeS2hjh40N1i1/kI1X5EoY0HubP+T3u5ecJP9lJX97sLvPeSumw9QyTUIxlPDFAFzQdnkM"
        "bgEKr5iEBiuRCzhY54AEYmgxg544SQZvJCPs8ntwFDBBsJ7CADIi/MRFyajDINQTcoKeo99GlNKM8LGfQ7vK3rXOpu0vpgyQOTII"
        "/lb2ybI2Bfk2/FQo41HAOlaPCyRUyZWwn1drK2uh9mXmkMBCGiMLME5aSqj3l69VncHlfK7hsP9+rTJlezr+gG/P+5P2q+d9XT1m"
        "UFfX+prB4Xx6Oj5Wp0NVrJoBX9UGdiS6CDlMhTEUkvDc3k73gULr5g5h0jUnSXf/Qih8Z4zE6FKpgjnwHJo2aiChu2HdpVkpNzSg"
        "NFuGAWkYXyKnrZ3Qcjs2iTw5y9sLYwYMyFZQYRC981p7Gv7zcmu8e2mS9ZNGNRSL9c78BVBLAQIeAxQAAAAIACJ8R11asgQb8AAA"
        "AJ0BAAAKAAAAAAAAAAEAAACkgQAAAABhZ2VudC55YW1sUEsBAh4DFAAAAAgAInxHXYbVzshOAAAAYQAAABAAAAAAAAAAAQAAAKSB"
        "GAEAAGV2YWxfY29uZmlnLnlhbWxQSwECHgMUAAAACAAifEdd+svChl4AAABqAAAAHgAAAAAAAAABAAAApIGUAQAAY29uZmlncy9h"
        "bmFseXplcl9zYW1wbGluZy55YW1sUEsBAh4DFAAAAAgAInxHXVQzy31eAAAAagAAABUAAAAAAAAAAQAAAKSBLgIAAGNvbmZpZ3Mv"
        "c2FtcGxpbmcueWFtbFBLAQIeAxQAAAAIADN8R11PoXQfFQsAAMEWAAARAAAAAAAAAAEAAACkgb8CAABwcm9tcHRzL3N5c3RlbS5t"
        "ZFBLAQIeAxQAAAAIACx8R10VXDhhXgUAABcKAAATAAAAAAAAAAEAAACkgQMOAABwcm9tcHRzL2FuYWx5emVyLm1kUEsBAh4DFAAA"
        "AAgAInxHXZkPmpd/AQAAMwIAAB0AAAAAAAAAAQAAAKSBkhMAAHN1Yl9hZ2VudHMvY29kZV9hbmFseXplci55YW1sUEsFBgAAAAAH"
        "AAcA0AEAAEwVAAAAAA=="
    )),
    "candidates/submission-VNEXT-INCLUDEFIX-ADDITIVE.zip": ("c26518e6a6b91eff4136238a121a93bb045e593e7586b314795b6945efac1126", (
        "UEsDBBQAAAAIACJ8R11asgQb8AAAAJ0BAAAKAAAAYWdlbnQueWFtbFVQW27DIBD85xT0ACSKEvXDl0EbvMGr8uqyluWcvmCipuUD"
        "aZbZmWESRJx03dC6PCOr2O4waY8xgrmZ6+VuSMw3iNlucPk0TpSiVIVXJ5TTpD8oubDOqAvnWKSe614F4ynOSknOoU5Ka6N5Tc2h"
        "iaZ5YITZPijggXAmeaONSfANPYqtArLWA9b1Hhu7gLjl972HtwnJL/fMLx4Cu8VWihSAD8Z/ehPyDGWIgMcktgfueftxOT3Id59l"
        "6qb2oNTzsQsJwv5EPu0Qw2uhflFpou2PTE8Y7bSeUKm2iAzSO07SfYb2n/LGoJUHsQRKfuj+AFBLAwQUAAAACAAifEddhtXOyE4A"
        "AABhAAAAEAAAAGV2YWxfY29uZmlnLnlhbWxLLUvMKU0syczPs+JSUCjJzE3NLy2JL05Nzs9LKbZSMDQyAArnJlbEl+Tn58QnJ+bk"
        "AEVN4IJA9fG5mXmlJakgYT1TmHhpUR5It4EBFwBQSwMEFAAAAAgAInxHXfrLwoZeAAAAagAAAB4AAABjb25maWdzL2FuYWx5emVy"
        "X3NhbXBsaW5nLnlhbWwdylEKgzAMAND/nCInkG44mL1MKBrbUk2KJrDj6/x7H89473wk84MjhuEFpp36n9PncYs4BtjTj9Stu5Fp"
        "YzkjvsP4BStVWpVMs8pacwTEKvPmC5MV9VzsnmvaToYLUEsDBBQAAAAIACJ8R11UM8t9XgAAAGoAAAAVAAAAY29uZmlncy9zYW1w"
        "bGluZy55YW1sHcpRCoMwDADQ/5wiJxA3uoG9TCga21JNiibg8Z37ex/PeO98JPODI47DG0w79YfT5+8WMYywp4vUrbuRaWM5I35f"
        "IYCVKq1KplllrTkCYpV584XJinou9ptr2k6GG1BLAwQUAAAACAAzfEddT6F0HxULAADBFgAAEQAAAHByb21wdHMvc3lzdGVtLm1k"
        "fVhdc9u4FX33r8DkYRN3JPkj8Taj7O6MEsuNp147Yzvdpi8eiARFrCmCC4CWtdMf33MuQElOO32xJRG8uB/nnnuAb65X2hulW6X7"
        "6Fq3cn1QwVVxzZ9Nu7StMX6ibk1wzZNRsTbKhtDjbytfvmxi7VrlTeeCjc5vlI7qaO38Y+h0YSbqG7YIfVEYU6p1bVpV27LEv2hC"
        "DGrtbYz4Vjkv5vouRG/0SlX2WXU6BDW7Ppcnux1eB2WebYi2XWYr+Nw0snyiPu+b70Ny2TzrIqpWr0wYYZ3Hh2g8PpvnwnTRIoK4"
        "6fgQK4JemoCUlMr1sesjnVtpWNsFX5pQeLvgC8ENu6zUk/ELHe1qcnBw+/VqfqduLtT957m6urn5cjBWczzfMJBmo1Z9iKpwbdRI"
        "pFbRuUYVGlH0LYJRG2St1kg4f0PmQr9Y2fjQ6VjUEzVDeM9x7NpmMLfWIdJpFXvfTrDX/gtqfn1+J95HHR4n6hP3sVG5tjAjpQN3"
        "88h4q5udI3hQIUeMyVYbyTUMoCwTdW3wozIlTKQ1Nk7UPWubnbGxhnGjQu18VMG0KDGwALe+cadUjcbCwSDwy1iid0MB1Bvv+mUN"
        "a+/GZ2pl2z7morw73jnJ/NewUKY9aUQjis2fxh9Ocr6ZEWRacGKQ8TJMVVEz7YVbrWAxpJd/+EHMh+g6eNg1zjNo5kiq8di69S4F"
        "l5VCnYHKvg2EyUgeEfaSKW+IFLjYYXd4x+YChCx93ghmVvoxQ5NpnM9ur77J9o/GYHtxvLINUrgBRIr8pnrSjS3ZYawgHQhM6k2C"
        "KTfs43QXFtadHR8fM1oPGwJ4NFf5IJbx9OTsGGVoYUV9BYaXKJ8at/APLo9b9Xo2+ti9Hqka7yBAbZvdupFqXVR+KYUHUTxZ79oV"
        "Kk03XFXR7BSFDZGuYm2NxNDZWxiTwD/bNpVE2s9V/w0BW6nOG8Lng8C1IlPAIJEgvUhclfAV7WiXAvuZ4HEfB4iYCAipQsgEwg1I"
        "/4a4RC7Vq64xGlGxGQEz82rYn6FpiaclATRTtXSAnrq7n39Rp+hx+XCisOv17Orbv+bqzYmg8vBAOqxwpXnYunFz/WmegKbVyfit"
        "5B2+/dGDqKbi26PZKAvuirayqJWqvFvtcc6bqm8lV0dFQ2LMbGa8J3eCD/aYbXi4cmWPUtdMNRriksAhAhh8TmLVkwy4g+CydFLY"
        "jnwifY13cgbV/ez2b/P7kbq9ublXn2Zf7+YjdXH5T/XlanaNX+dXs/v5+Ujdz+/u70bq0831xeX5HGFPETJo3aEz6EmCTvR9BJXd"
        "PdoucRHgbau9eAdPJZQ9Fk/gRemGfAyEVGRai2tLsknlOUV5xJPbX4WKUwyoFEogDHII1OyagtsUNgJT0hfqDeDm4wO/HAEu8gHM"
        "uEDLqx+PxyfHuYEOxSNAqLI+FY3VVwtDEg/k2GFklITpZbWdaV4QZlZd3CgUcs00sQO1B3GTmINpqgScvSn2HTimuXfBdK90W4B2"
        "X6nx2LZF05fm579Muo2aqH/nRU9pOB7hB3a2Gp8eS3O0kgcxWhpMA5uSO4uAEYbV28EpSdteFKlcya3WgbFWC4fxAQvJWIvXsKEK"
        "dtGQHb/DcY3MNcN8CbDCNBYGoTClwDXblGUdHq/1ZqK+eFOh5pqLg/FPINcnVqYEnVrOTZgFhbgnWQQO3lLEeUI4h/bSxAfpUkzL"
        "pdddnfiNO4Gw8Kb8KOMmJITu1SHFOXRaKxAcRp2k5+7lJNEUVmC6NG+RRXAZWGhdW+R0yAmxIBCQ4Nfkn5TCdcKSBUREOiy2CH8L"
        "hM/PL4FpYA9PZKAsNmmo/xhTvT7kSf0+VU/iIIbhBNeP0lsccS7N4gUrZp7IR4Vhj/w6DKywoiShekkpjnQSQzE36Y4cONY7DwZK"
        "kTHRqfV/ZwBiagPYu9UExilTCBQ4HuKmAfn+nZOw64GZQs2+XGZF1pS7RFA09q03jWbD2hZDMOFyBUqXWSSVYtmGRCasIu8QAUzq"
        "ljH3pWDCJbfLk2D0UgkCZgXX7jYdWqFxS3hbo8QgoZb0sWmLI82/mN3eavAwaHpT4gMWPp0cPZ2qhWfLipaEJGq0Tw6AVEbQwyWC"
        "y4KANkE3gRv3GHx+4FwG/ZltBNwukBpoeDBUW2JHg0CvHUkrUQwUBJk+YYyadyQ5lTgkvCyms5ShZUJD2JHSAkyC9Q+gc2J6yK+w"
        "kcY/E2phkVHqlNM86CgJZMCinTllrJBe31qMv1xpowUB68F0CnBg2mxFBFrXEOB2uSTnCwYDpQCYh2mChqXDCJu9H+yfSNZOL7Ll"
        "OXragpiRum3DUxUUDmrWUp0Rj85vTcoPUhFvxluaHIQsXZN4s3CimxiafjMM/NJWYCvD2bdyELyhMwWmfLGXyw+ZQkTwlDu2o1PM"
        "R1qV5RabFQv7Zm+g126dOpCbySwPnK1LLOUvrCRPWybFKmyGnBV4GwDT0g1ZLmSVyzxhEQ6A5Zbg8IY3YkaoYziG0aRQXuElcykB"
        "aH7FoddB50E6AbxHcdWpJ6upmx+yTlVvUAv1izw7kpc4rn766fX85uK1mkyg428uIF5m7UZEZHaZmA2gp73DJpgBNlELdDXlYqLA"
        "dGaCc3vHFhmAI5nX/IgN2ZXyEXOPn+HG75CmE9BTw3Ec+25SVJjM0T1zjXqDLG3k9MKKkZArVhYTo4STh4MoQaBSAuiIUI/SWmxc"
        "m+KRyHZsdoNUgnZ2gVAwUd0LQyfiSg27ckJra9ejZxsXzHB0ex6mwTtMg3/Mby8vvjGvRHxqoffj0/eHB/pQ3fFA8bxXIdvKWY2L"
        "p0wCj/PjFT6xQB2xcnF5NefxKD98WajTX344gZLg6YBKYqI+9rYpB3kD+y+VCg7vobVdZ6DpQ9Z+SfZ5k6R2CyQtDtW1hrBdS6WU"
        "DLaJurCiBLZip4Ha4fh9lU/7g545m6jbvp0eKPUiHjGVpA//PjzT/fHjYGP8rMZ/QDXFxc/p3Dru4M20AC8xDo5C/zLat8cHxSEO"
        "NHAnBA4OFDtJ+XXt2EfJd3ZblfgtN5jMf2EuoD9LntQLw1wePx4KOe2v1MNZtzZNx4kFmAf+J40LklAvHfOgddQ9eXxS+mieRDG5"
        "GhADQcma6/J3oK2Ne57KNIc2ChT/B+WhuoCh7cTNdDuSt4fcFgnOCj5tx48pl4RUQC3NYbpwoNji4a0P6VCdGwbTrFMFov2NVJlY"
        "/69nw1FdNaZCvwhShqPXGa9XjLSZsCPiClP092/SIjLgGH0e2bxeoehCpjjcp6ljthElZt7T0xBs6HudtO/tHtnnU0nm5q3BD+mq"
        "6qXOySoEKJrzeDaCiAZ9L/po8ve+TfsYnvg3aHyU1y97dnf2MOlpjqCheNnX7xqKTnLtZJuBl3daNJFlzDTPtV00Mo/zEKeBi5RO"
        "le6kCAoCsLRlOrK5vqhHw0yLHthZaBReMJPmQyfCBb/I7U2KZMCvzMnpTrHhUL/EyVo8v7DPCZ9d0y/JSEQzZGZKqZxxpy9uGeT2"
        "6MP2gk8GS+IbvPodMpMaH4Yp45Mj4jTd9ch8kAmb1Gh4WXY5x2FNvQeRlqO9c3KWFc2Uxme96XhsCDbsn5p0jJReQfpjb0km7DMQ"
        "9t3Xj79e3vMm5/8T8Xz26fMDtf78/GEg5Txc2FbjsRDXBGZ4mB5uRwKKUBhp73TxoUG/2n9QdLyl3va8Dcmjexjlcp8nR+r9q8R8"
        "UPwfl3uYz1lipjtH3eCcFpKmmpLi03qikhdnIt2DlDzZH/oyxHHoO7YOXF9h0uK4IS0G92q520E7b++T/gNQSwMEFAAAAAgALHxH"
        "XRVcOGFeBQAAFwoAABMAAABwcm9tcHRzL2FuYWx5emVyLm1kfVZtb9s2EP6uX3Hwl9qDLadZUxQuisFNnMVA3mC7awMMMGiJtghL"
        "pEZScYR1/33PkXISbN0+tGb4cnfPc8/d6cE0JKwkQVaKfGR02VJmcklKP0rn1U54Y2mLf76QdN/6wmhcrY1TOGhJeBofjN27WmQy"
        "pak2uGdJ7KT2dFBlSZXYy/B4q54+Ugt/W6Vz+no1W8xIhNVDSg/Yrxrn6fZuRZXJ1bYdUoaYvCQ4z2UpsRK6xetSDsk2mjwCdEM+"
        "Vtp5AV8494XSuzRJ5svllxmt7mh++9tsuZr/Ol3Nkj9razalrNa5dJlVtVdG//JXklxNFxd0Pb+Zr5bJiKYeESCU9+SNKSmDZQcX"
        "+AtOhoxF08EqH2ExGdYHBBbUaS+fPClHrsK7CdVWgQnRmRQb08DwCZVKS0c1qGLz1C/A/pCczGmk6c10+Ll+M0jpi5MhL2sGDTp9"
        "QQBq/Zqfj6XOw4L60eyHo9lKPA1SAOHnO8Q3JKSF7I73nBQ2K9ZOVaoUdh1yzVnYSR/+WGupdsXGWEeeM7d8uPl8d02305vZkvos"
        "BPkkqhrh9M6NdqaUaYDY4zz0amGdXDMYaXsD+JWPgOggBqkz6SKiH4UQsO0/nf1XLOEcuF62Pr09G4bbQbMG5oO49tocsM1RZp5c"
        "W21MCTnczFZXdxfJ25TuVban09E7XMoKNr21pgqZVM41kYyY1z8aCGzSWVI5QKitkhaSk9ZyRSDTQwJkUUGcljR+cVg3m1JlNL2f"
        "x500OU1pGTCTg0gQKIc8CamhkdXUi6H0aDRSOiubXH76Ka1bSul7d+mRekHt4x62mF4anZ6kdMmVlEsUlGIpdzS92MVRiOH371kp"
        "nAvrH3phkynNtywULiBQ7bOC0XjL5CJpCJBBbq3YVVzcIMC12ui2CiWIEmHW0Ab2KP6YRo4DtdxALB0TP6ev9Mz3X4I/qvj9yehZ"
        "yMKaBoaUh5zfhfj4zUYW4lGx/1oAs2ONiZK2jc4CC8PYYbiwkK3XPHMU/f+lObL8TDJrDI7+Lcg0OUvp2mSi6wLySaFbgjg2EHrU"
        "s9+yc9yLxp9tn0GWq7ZWiJNKAe3irA8oaH0xj6ULzaalR4EkbPALOAA4mNClcJ4FtpeydlSaHQSHDrXFtqjVGLRxMAA3BMM1GgXK"
        "T0k3xnbpwnaQbVwaXOBXL4cAW9XChzVeArmNB6KuoW0RaMbGR1ooqBqeLX7H2bEh4CYXR1g433ZblbD7po6nYtNtOhnkFNet9uIp"
        "hmcN6HOdk1iKoQk7m4270nRjiEt2ITtc7qIa0gsSkYvaH6PPjNmDhWDzarW6/8YGC+/rp/E6K9UxjPUrs+vGvnDCXYyvpLQK4VQq"
        "9IHXEifoVdpORjyGtq9bi9u7MEsF3c6+0hbTrbFhvj03kSGtpotfZyueIGFodqLmS7GGvYHYEEgeu99ido3JdtF55ydZaRyL0Cn0"
        "IShScbdmjkPa+H0Xd18w49lY8P/kD0oHsbMNB2vQRfSsEe9R+CjEZDG7v1usqI92TKcnJ4T5n3PfKwUPydAUj/OOO0fo706rupbY"
        "2NKHWNyDSRKhYkYKX0zCYBsBV4B1xD0OoJPFHT4LzqdflrMJHQpYD0DZcG4Am3s+F0fjXk4j5SwVBSFxtHaPfG+lDaMIySDXZMUg"
        "uZx/o/vr6e2kK2Tu9lkh9A6fGY1jBrvG7tROh4y5ECL2HFrdPwdI0mVkQvFTCLSwOy1lHvrDkd3ogvrMO4x33WrYvYqacgNOCJqs"
        "TFb4iFlOftBn+rgUZhZvdCow4fuon5s4+PlbqZDVIDm/u72cX8xuz0FjgYTSGChy1VRYlOYQFXUo2uRvUEsDBBQAAAAIACJ8R12Z"
        "D5qXfwEAADMCAAAdAAAAc3ViX2FnZW50cy9jb2RlX2FuYWx5emVyLnlhbWxNUU1r3EAMvc+vUG8t2A5LQg8+FJaNUxbCbtl1oTcz"
        "sWVbdD5cjZzU/fWVvZT0JJ40896TXrAeS2hjh40N1i1/kI1X5EoY0HubP+T3u5ecJP9lJX97sLvPeSumw9QyTUIxlPDFAFzQdnkM"
        "bgEKr5iEBiuRCzhY54AEYmgxg544SQZvJCPs8ntwFDBBsJ7CADIi/MRFyajDINQTcoKeo99GlNKM8LGfQ7vK3rXOpu0vpgyQOTII"
        "/lb2ybI2Bfk2/FQo41HAOlaPCyRUyZWwn1drK2uh9mXmkMBCGiMLME5aSqj3l69VncHlfK7hsP9+rTJlezr+gG/P+5P2q+d9XT1m"
        "UFfX+prB4Xx6Oj5Wp0NVrJoBX9UGdiS6CDlMhTEUkvDc3k73gULr5g5h0jUnSXf/Qih8Z4zE6FKpgjnwHJo2aiChu2HdpVkpNzSg"
        "NFuGAWkYXyKnrZ3Qcjs2iTw5y9sLYwYMyFZQYRC981p7Gv7zcmu8e2mS9ZNGNRSL9c78BVBLAwQUAAAACAAAAFBd+svChl4AAABq"
        "AAAAKQAAAHN1Yl9hZ2VudHMvY29uZmlncy9hbmFseXplcl9zYW1wbGluZy55YW1sHcpRCoMwDADQ/5wiJ5BuOJi9TCga21JNiiaw"
        "4+v8ex/PeO98JPODI4bhBaad+p/T53GLOAbY04/UrbuRaWM5I77D+AUrVVqVTLPKWnMExCrz5guTFfVc7J5r2k6GC1BLAwQUAAAA"
        "CAAAAFBdFVw4YV4FAAAXCgAAHgAAAHN1Yl9hZ2VudHMvcHJvbXB0cy9hbmFseXplci5tZH1WbW/bNhD+rl9x8Jfagy2nWVMULorB"
        "TZzFQN5gu2sDDDBoibYIS6RGUnGEdf99z5FyEmzdPrRm+HJ3z3PP3enBNCSsJEFWinxkdNlSZnJJSj9K59VOeGNpi3++kHTf+sJo"
        "XK2NUzhoSXgaH4zdu1pkMqWpNrhnSeyk9nRQZUmV2MvweKuePlILf1ulc/p6NVvMSITVQ0oP2K8a5+n2bkWVydW2HVKGmLwkOM9l"
        "KbESusXrUg7JNpo8AnRDPlbaeQFfOPeF0rs0SebL5ZcZre5ofvvbbLma/zpdzZI/a2s2pazWuXSZVbVXRv/yV5JcTRcXdD2/ma+W"
        "yYimHhEglPfkjSkpg2UHF/gLToaMRdPBKh9hMRnWBwQW1GkvnzwpR67CuwnVVoEJ0ZkUG9PA8AmVSktHNahi89QvwP6QnMxppOnN"
        "dPi5fjNI6YuTIS9rBg06fUEAav2an4+lzsOC+tHsh6PZSjwNUgDh5zvENySkheyO95wUNivWTlWqFHYdcs1Z2Ekf/lhrqXbFxlhH"
        "njO3fLj5fHdNt9Ob2ZL6LAT5JKoa4fTOjXamlGmA2OM89GphnVwzGGl7A/iVj4DoIAapM+kioh+FELDtP539VyzhHLhetj69PRuG"
        "20GzBuaDuPbaHLDNUWaeXFttTAk53MxWV3cXyduU7lW2p9PRO1zKCja9taYKmVTONZGMmNc/Gghs0llSOUCorZIWkpPWckUg00MC"
        "ZFFBnJY0fnFYN5tSZTS9n8edNDlNaRkwk4NIECiHPAmpoZHV1Iuh9Gg0Ujorm1x++imtW0rpe3fpkXpB7eMetpheGp2epHTJlZRL"
        "FJRiKXc0vdjFUYjh9+9ZKZwL6x96YZMpzbcsFC4gUO2zgtF4y+QiaQiQQW6t2FVc3CDAtdrotgoliBJh1tAG9ij+mEaOA7XcQCwd"
        "Ez+nr/TM91+CP6r4/cnoWcjCmgaGlIec34X4+M1GFuJRsf9aALNjjYmSto3OAgvD2GG4sJCt1zxzFP3/pTmy/EwyawyO/i3INDlL"
        "6dpkousC8kmhW4I4NhB61LPfsnPci8afbZ9Blqu2VoiTSgHt4qwPKGh9MY+lC82mpUeBJGzwCzgAOJjQpXCeBbaXsnZUmh0Ehw61"
        "xbao1Ri0cTAANwTDNRoFyk9JN8Z26cJ2kG1cGlzgVy+HAFvVwoc1XgK5jQeirqFtEWjGxkdaKKgani1+x9mxIeAmF0dYON92W5Ww"
        "+6aOp2LTbToZ5BTXrfbiKYZnDehznZNYiqEJO5uNu9J0Y4hLdiE7XO6iGtILEpGL2h+jz4zZg4Vg82q1uv/GBgvv66fxOivVMYz1"
        "K7Prxr5wwl2Mr6S0CuFUKvSB1xIn6FXaTkY8hravW4vbuzBLBd3OvtIW062xYb49N5EhraaLX2crniBhaHai5kuxhr2B2BBIHrvf"
        "YnaNyXbReecnWWkci9Ap9CEoUnG3Zo5D2vh9F3dfMOPZWPD/5A9KB7GzDQdr0EX0rBHvUfgoxGQxu79brKiPdkynJyeE+Z9z3ysF"
        "D8nQFI/zjjtH6O9Oq7qW2NjSh1jcg0kSoWJGCl9MwmAbAVeAdcQ9DqCTxR0+C86nX5azCR0KWA9A2XBuAJt7PhdH415OI+UsFQUh"
        "cbR2j3xvpQ2jCMkg12TFILmcf6P76+ntpCtk7vZZIfQOnxmNYwa7xu7UToeMuRAi9hxa3T8HSNJlZELxUwi0sDstZR76w5Hd6IL6"
        "zDuMd91q2L2KmnIDTgiarExW+IhZTn7QZ/q4FGYWb3QqMOH7qJ+bOPj5W6mQ1SA5v7u9nF/Mbs9BY4GE0hgoctVUWJTmEBV1KNrk"
        "b1BLAQIUAxQAAAAIACJ8R11asgQb8AAAAJ0BAAAKAAAAAAAAAAAAAACkgQAAAABhZ2VudC55YW1sUEsBAhQDFAAAAAgAInxHXYbV"
        "zshOAAAAYQAAABAAAAAAAAAAAAAAAKSBGAEAAGV2YWxfY29uZmlnLnlhbWxQSwECFAMUAAAACAAifEdd+svChl4AAABqAAAAHgAA"
        "AAAAAAAAAAAApIGUAQAAY29uZmlncy9hbmFseXplcl9zYW1wbGluZy55YW1sUEsBAhQDFAAAAAgAInxHXVQzy31eAAAAagAAABUA"
        "AAAAAAAAAAAAAKSBLgIAAGNvbmZpZ3Mvc2FtcGxpbmcueWFtbFBLAQIUAxQAAAAIADN8R11PoXQfFQsAAMEWAAARAAAAAAAAAAAA"
        "AACkgb8CAABwcm9tcHRzL3N5c3RlbS5tZFBLAQIUAxQAAAAIACx8R10VXDhhXgUAABcKAAATAAAAAAAAAAAAAACkgQMOAABwcm9t"
        "cHRzL2FuYWx5emVyLm1kUEsBAhQDFAAAAAgAInxHXZkPmpd/AQAAMwIAAB0AAAAAAAAAAAAAAKSBkhMAAHN1Yl9hZ2VudHMvY29k"
        "ZV9hbmFseXplci55YW1sUEsBAhQDFAAAAAgAAABQXfrLwoZeAAAAagAAACkAAAAAAAAAAAAAAKSBTBUAAHN1Yl9hZ2VudHMvY29u"
        "Zmlncy9hbmFseXplcl9zYW1wbGluZy55YW1sUEsBAhQDFAAAAAgAAABQXRVcOGFeBQAAFwoAAB4AAAAAAAAAAAAAAKSB8RUAAHN1"
        "Yl9hZ2VudHMvcHJvbXB0cy9hbmFseXplci5tZFBLBQYAAAAACQAJAHMCAACLGwAAAAA="
    )),
    "candidates/submission.zip": ("46e455c81d594b9532cd51f142bc7482020293514b36d6a539ecff21edc67343", (
        "UEsDBBQAAAAIAORlSV1IfImXTQEAAB8CAAAKAAAAYWdlbnQueWFtbFVRPW/jMAzd/SvYvUoRXNEhQ4EO3boVhxsFRmJsAhLlinRS"
        "//tjnaC9m4RHPvB9CEcSi6mg6gHeSn35woNgpQOMVCtGvVDEjLPxmSKdOZMkiuf9kElTZ583OcDzAPB6W4bcnSvw/ucVOs3IHfDr"
        "LJxaB/rEZPdQWbhigWVW64Q1pFZnND4WAn/TRLrzk7+VFAp9csJyD0oVxTgBSoax4zyBtVYUmpQVLpNr2kSri2orZwKE1CR1MoLF"
        "fXVDFlt3Q22Zyi1feAy/9sfAFj7QwuUR908h2cDivpZ0DXfHksqS3Vl3l6YPuqpR3dU8bPoHdxqgLxI9hVvMV0yY44kLbYgy2w+6"
        "dDb6gSNZVENbdIO6HKuztx6+98lNRyEep2PrNx5hT1NUrlywb4z/6X5oq2nw+qmjS3ohtv14kxOP/0S7Djwa1rmwjLsVaxn+AlBL"
        "AwQUAAAACADkZUldMnwHi2oAAACBAAAAFQAAAGNvbmZpZ3Mvc2FtcGxpbmcueWFtbD2MQQ7CMAwE73lFXoAKAkTyGau0JolC7Sis"
        "pT6flAO30WhnwVvjPsM6Rz+dLg7aqB0Ybj+u0V8nt807qaEZCFpZPtE/zmGsc5FaJNGi8iopOu//6mlrYhx5uA9fZHnbyoSsljLG"
        "A7qx+wJQSwMEFAAAAAgA5GVJXdgxSiFNAAAAXwAAABAAAABldmFsX2NvbmZpZy55YW1sPcrBCYAwDEDRe6foCFVBpMuEUHMopAnY"
        "RBzfXPT4H59uZEfrKjXlbH2QusGkpnLOmrdSggc+YKoMDZlD9x/jh9HFjYKPT/2SyGUt6QVQSwMEFAAAAAgA5GVJXTe8LnAnDwAA"
        "/SAAABEAAABwcm9tcHRzL3N5c3RlbS5tZIVZy3LbSBK86yv6NuMIPmR7NeuVYw60BNuMlUSFRI/tvSiaQJPEGK9BA6I4sR+/mVWN"
        "hzS7sQfLJNFoVFdlZWUVvpetsbUztjC2bcqizMvWG19umwN/dsUuLZyrzTZ9SoudWd1EJvW+dSYtTLN35vbY7MvC1K4qfdqU9dHY"
        "xswPZf3DVzZ2M7PGotZjh9x5b3d8UoJ/R9NY/8Ps06LxJnFbPEX2q90frfONS8zG7e1jWtYz8xW7mXK7zbDo3CSlKcrG2DjGhnJP"
        "4ZqDLKlhlW9slhk8+wce5mfmvtWFubOFN9bkaZHmFivqMmnjJoXxlW3iPf5673TDfZokrjCPNksTK0sa2OTF9Kp2OM0jVrqn1Dd0"
        "Sm/pycnF6i4yd1+uovuTqYkeHf3hPdbZojFNWxdYvC3h2LjMq8zJ3nnrG3wvGgufWtOUZWZiHGJmVkVMj9jkOJFfjG83edo8qMHu"
        "ycZNdjQlV1m1HH7E2UZbLLYNfJ822IaP9xOzSx8d7nHG78u6GRviXdE42QznDG6Wx1p82mMb7jvDwe5gkjyOMfzJm88SRe/UnYc9"
        "fCduKhrEH/Y3Ic4w0j2miTyjKQ3ck25xND6nLSTuuN9uMlyt22bPR904rDIuwQkkBhP5j3DEcRy+wnFb/jSrsFN1lI8I8QTmNG01"
        "i7c73FI+6W/VEWH/HXbOmjLPsCZtXAz3l3n6p5MdCNXnv8yqZo/nZXwYEEboBtDzRxie0DO0ba7GzWUdPwEccyAwriVesgfin5XA"
        "DEI9b/JqZr54Zw417HiQ7coCEYVvYroNkTA7V7TAPX4tHIFsAakRdnN8yhx8KH7CjcjEzEoMB4ByZ3FmiVSrXMEIpDxPlzmaTBOz"
        "qUtEtnZbIKus8QOwmtuGm0zFsnhvix3v9BUclyE5AKbF7XK4gKO3Re1wCTnMuCEHb0POmMLmcnO6K2yIH3LfthkD655iV8mh4kxy"
        "cR4oA9cUvaY5VvKMtqnaJhiH78SrPxbx3PLviDlu3AF5XeOpyAJPxLfZkMKmzAaaMZtjZwsOkHXcIixFUkpxk5E0OKRevLkAxGyN"
        "wO4BTmywq22FrEzAcSmZJkOWKLgRr3I7M5fYI2YmDvGbNzWCtQFbDZmB09W2+OF56Dp1YCE4LE8zC5QcsUtIzMymeeAloa6kMztQ"
        "SHASznyQoxzDMpr+T+eqsABeyYUpLso8hyO9gT8IvLPJ6enpFHGFheSQsF9sq/fCSYrXQCtk/bxEYr4+OzXkaT9/fcoNTL+B4mDr"
        "6ChXzffiHeUgGl8H/DCWja13jviR480CCSRtXoFcysx16ahfxO+KPP1BGMK3yCkPSr6MLpb3y9WNuV8v1tHJNVhWmHZbtrWwR3A7"
        "khQw98hMGAsWzY7nZr24+xStJ+ZutVo/XCy+3EcTE32LLr6sseHD7WL9eWLw9+Kz+dVcrb6aubmOLpdfrvHh8/LT55n8VYLfgHDb"
        "qsJxWduOzM9yIzBMDNNtZj44D1MCjde4hdmjDrHC4qSercXR1TE8sKZdYvbHithEpSHx4Q4SPA4XA6t6cImKsSSf/jF9nTg/0TXr"
        "X2ezmbmTv9/k7638/beJflteRjcXUff12210sY4uw9eb6JvcehLAWdi6Jn3VAA0fC7z4siAVwWZEnyWhLTrjeXxGanG5uF0vf0MB"
        "XcHBN59O7gGExnxc3K+1qNBmVR8sYoL0Ln0mWg5BA/mmBLMTIWSbQEg5S/APKS0Md5NuU1e/H/LZeHnUzeruenGlpHwZRbdKx/Ls"
        "HMSQVryf9TLHHqIMmKU54QTWbX26IRf3lBLXpWgKJWmGZiApXkbpxSabNENim+YA8aKcPQlSCDyBOowN6FP4EFzDC74l+xJHcZno"
        "CQfQDEgQrHimOz14LpAoNzAMAu/5IQRjWQkkBEC9N/axTMkoQ9FReiNaZE911bkegoIGe1AtKVsHfVCAIIlU1APwWKE1XCGNw/IQ"
        "siG5Od0qSakBcneW/mDdQ63nAxmPc2RtlpUHuYxwt7AY/Ni6OfVTDTRvedWK4we3x3sHglUH89nPHb8B9WBFxz1EFMKA8jaoGU8/"
        "xQNFkzayFKg1hxTqoG6BqNyNhODFviy9ngJb24pOKNxToylntXY3e8vzF+Mzx21dIyyjMPaMj6cYJ5JSNkH1676iRiF9rlYXi6vl"
        "vxbkJnO1uLyM7k5ez0z0xCM1IT/cU4BOD1J1SKGLEtcQNOddpR6K57hSD3V5qNnPS7MEhddZDWPrycRvZma5FRLMUpeMUNIn8RxR"
        "2YtBPOtIYHaiHWcgcIn4Dlq9yBFKDOoap8zKWrQ5FU12nJ28hRDTcq2OGGiAZta1SLanRgSlyHXiod61TJLOG1izBW1J4RaNohcU"
        "BFLVdpAC/EC3x1mbhGIES+k5BRMTTSweyTjpgFIhlNnJ32aDwu7TarREPKRCUWgi5JYHdErIUW4tJThsI6UgAxx9M9QNyTq1ByYn"
        "qbf5Jt21zF+pgNCIoxbs5Ew5UQXPQ5AjD6SfEUWqYzNEhEzCpIKG7BoJuKDMHkPdkfpOOzwRjMX/j1QBHbgPLQLgHlTchiLlR1Ee"
        "ihdBmWveh1Igmr3vdZIx/6s4s7gdPY7WlcT5uE4V0QQZVarNphnys2XzynwQIhQsD20VdmLrgJ1GRMgdBFlk/qassIKxSUW5whGg"
        "aJGXDOeWfZboQGh4QdDJL+pzeEoc/VC4dLff0KXictUE4TRHbmyl4sOi7uygcWx/2KfxvuPZufwPeTZCnKHEd7U+bsSsSdmQKYIf"
        "t3WZd3Up9Pqzk7+/sBENqtaJARWWkLCA3ZsQLd3Py9gBBheBj45aLqchevCgfc7q8Bluxxn2aUWFjaBjY5h9y0x8Mz17JgH8+9AT"
        "hYkCBTlhh9u2JTo8PFIDefJOzwAx4XLoM9Y6tmbPdPoYRR3b4+omwB/F62WJ6Os2BXiIgdCblhbSySBnAilS0WB3bYED1006peCE"
        "M7q64Loc6TVXkFUoJ4oJ5mBQi5q289FYAOANbUIv4Iue7uYeNSu3il0SYi19iyAjLNE6ygZF5w9dn98Wvcidw5tNJxuGVsYycPYR"
        "hlkRS5KCo7mA3aCE0Le4MR+PViCs1+YTJXzEaUAvCAOZiD2lTCxaRrN+obd7sa2aSUr9UGCGei8xAroEW8YVZbvbMyd/h4hHnoWS"
        "znrd9zNW/MzovIRMX9neS8oDQfgFWdzg28HWnegSEMAmnYigOI7Jvspa/9zU/17HKSQ2Lljci4aDDXRjqaN4fBJN0eYb8t9o0LND"
        "kvipDFqsOSBoVVcfRIqVO4A7CLJMxLfLvMP1Gr3wh1B1cTO8MOlJ6dz8bF91AaIin3fy/OfNKyNq4tgfgMmeyJFxOX41mhSRKCUs"
        "cHuShRmdj6Hv67TE4uTVS6eAoJqpTsm67bHOvdJel2zQhfAji9CgykUjTjhn0/peIiI17MBv+ORh3JSyFUGKR7oRVQHgmuNfarP0"
        "TziusmmtgiFpkdkxC/iLGtfJUvhQh2s8FOQEY+/iIGzkrLVO5IJKDIVf5aWlrHTJcLOIZdIZg6EtuqiTjrzaIgUdcvLxACwSs3FZ"
        "EXFC7wAJ6vRepRmfGfBPHlb12sdWZIVKmbLoqqF0HCNlFxQBxcj4CKGKcE5Ff7wfNIJQ2HAYpa/A4PKQ5lByfiN97WgLbXZkfMej"
        "Msqj0XS3ymhx74ZvDM9jKpSL6iXTjwnHta5WQbnXCZGcvIyhzP2zmWufYwSJjPD+MnRT3w/jPdDYb9Hd8uPyQlQ65fldy4oIUDzN"
        "U5afRmEBf1XgkgqIamRYOvBOMhaOuusb3Ucc1yG814XaRxKKOn8KA9SaHpWK301bdHqqNkPPxOg3pcoxEfs+QTyvKn+iPYjWqfMT"
        "Y8zt9/Xn1c3l6mb99W65jj58X0cXq8vo19fcG8F4a6Z59xhmNErSnF/UOdXRTH+YnzrDH3jlJzN9MtM/zHTabH5VCE8rWHcu1nXG"
        "Ud1LSfN7EH9i9i4Dfc2rdoPc43wS5FK7kKedGyd9l/GsPdXAUWxwXpL8DuwUXZuqC1DtN20iRYfNKHT2Qrpl6sipzJ3CXO7lHFFr"
        "Jqu2tCa6hxQaQGpUYHLpAS6EHnrbNtT2YDyk0s83sA1dixTYrtUK7dmckBWS53j0ZYfWjVQnL0nv1UixNWUL30rVHFU79gDwsZYo"
        "Tfsu34UraXvad2Da/T/pEOvcfL1b3Xx66MZp+u3++/WH1VX37eVYTX8dD96WNxer69uraB09fFx+m5j76Hpxs15ePFwv7685hJsY"
        "LkBmfVheLdffH+6iT3fRPWd/wsTRzW9LbHod3awfPi6WV1/uIkROQxqyPhyprQUTMsLmLD6knbYPquqY6cVjWpeFEEu4C/sVRtN4"
        "Hl5RzFG/d+mw7zAcD0lNBzH5WOBye5S+5n9svtwO3PhyzsMoxhQRG6CslK5O+gqAAHlRuMPojkEaP+dbmiL5aOIjYGmQPYU0z2hF"
        "lmFeww6XWrdVRk/S7ZYvqiiCu7bxBT357ljBj2xLc3Am2Xj0YsTGsdQWTl7D649EmHr0MlEHwfoWTLwkGUAbVKDL6xeZ6e7kMODb"
        "D18ugTlzu7paXnw/4dtIvd6SJ0ASU8nMDB1toy3JnrosdiLovOpTZUh/bt5xEqHi/ZfTYXyKr6/fnMr7PfhWR69STkSUkWVEgYUR"
        "+dvTU74rg9oJI6dQmyUggRym0mGIVSwE7A0gsY+VqHoUuXMTLe6/m3fT1+86C3QgZ16/m749m5jPi7tL8/ZsenbKifX6LrqOegPO"
        "zjpyYd/WBVPaSdGj6Z/DVDBMnuX1Kexig/sXBoArpZkpWCUtRZLEEDQpL5coHuA1N0yyNnyXI2ZzLNDYvtxqNyQzYmoI/jwaExMm"
        "M/OVD9K3bv8YOfL183jIYEvmQGlMVlKuheYuVKfTz11brm+9dGZkVYB2lUH8RYaVlkwF26h56YaYXEoUhldRik/+uMHxpkMfUkA/"
        "KmZFnAKeH5c3CNr9lw8gMTJVp6XHb3rPxRwVCcEGeD7whibPQVViJVoTpr2XW0a5Op0Kxt73Ju9k7KOJw9dbOCS5BqTk0MPVbhgw"
        "/fe8fiIT9m95L/7ydpoKaYKWmyojhF2bCwKKvU/3bnp4EcG8rHOv+3gvgQjzxOF9df+SeowbfeTBhns5tuhFpb4o01f60/BqSF53"
        "zk7+A1BLAQIUAxQAAAAIAORlSV1IfImXTQEAAB8CAAAKAAAAAAAAAAAAAACkgQAAAABhZ2VudC55YW1sUEsBAhQDFAAAAAgA5GVJ"
        "XTJ8B4tqAAAAgQAAABUAAAAAAAAAAAAAAKSBdQEAAGNvbmZpZ3Mvc2FtcGxpbmcueWFtbFBLAQIUAxQAAAAIAORlSV3YMUohTQAA"
        "AF8AAAAQAAAAAAAAAAAAAACkgRICAABldmFsX2NvbmZpZy55YW1sUEsBAhQDFAAAAAgA5GVJXTe8LnAnDwAA/SAAABEAAAAAAAAA"
        "AAAAAKSBjQIAAHByb21wdHMvc3lzdGVtLm1kUEsFBgAAAAAEAAQA+AAAAOMRAAAAAA=="
    )),
    "candidates/submission-GEMMA-HYBRID-V2-ADAPTIVE.zip": ("2932140057fca9e0aeea2dad736a57dd025fcf6ab1f5f81c8e49d63274e7d66c", (
        "UEsDBBQAAAAIAAAAR11SglB4kAEAAJECAAAKAAAAYWdlbnQueWFtbFVSTa/TMBC851cs97hVeU8cckCqROHyHiCKxNHa2NtkJX8E"
        "e9PSf88mgffg5lmPZmfGxoGSWBew1g6eQjwuuEkYqYOBYkRbb2QLTcjFXt/ag0WPk/CVmpg9hT8s82geDr1hMT9RzO0RD++Mk8ZT"
        "dYWVnlMH7xuAM6chkMFlC/xVgvOPE2wrOvh4PH/ff/7y7fn4tP9wOn2FZU0FN+ZKCfo70JU9JUetymF6gWZAIQ/kWaDMgVqYSu7J"
        "9HTJhcw6v40qISNBnetEbuE7VQefdUPKooplTi1UipiE3X4oOI2gITDgEgJyCvdNJtAv1rFaUGulwgU5tHA0n9bTXEjF1lqpthA5"
        "cVTyhOJG3YXJA+o1aYA695Fl13CqUma3dfWGkwuzWtMQcZK6r/cqFHfRN5JzqJ2qm8WsdVnrT37DhN5eONCKlsiv6FZY6BUOJLYK"
        "ylxXuJmwq7+X+6Ubm4iHsdeEG4+wuNFWjhywrIz/6Sq0ltboC1PRJ9FxkvWP5XTh4Z9o20CjYZyC/ovdHWNofgNQSwMEFAAAAAgA"
        "AABHXTJ8B4tqAAAAgQAAABUAAABjb25maWdzL3NhbXBsaW5nLnlhbWw9jEEOwjAMBO95RV6ACgJE8hmrtCaJQu0orKU+n5QDt9Fo"
        "Z8Fb4z7DOkc/nS4O2qgdGG4/rtFfJ7fNO6mhGQhaWT7RP85hrHORWiTRovIqKTrv/+ppa2IcebgPX2R528qErJYyxgO6sfsCUEsD"
        "BBQAAAAIAAAAR13dKCkzTQAAAGEAAAAQAAAAZXZhbF9jb25maWcueWFtbEstS8wpTSzJzM+z4lJQKMnMTc0vLYkvTk3Oz0sptlIw"
        "NjAACucmVsSX5OfnxCcn5uQARQ0RokAN8bmZeaUlqSBxI5hwaVEekG8KVAcAUEsDBBQAAAAIAAAAR13rzkCvxg8AAC0hAAARAAAA"
        "cHJvbXB0cy9zeXN0ZW0ubWSFWmtz28YV/Y5fsePORKSHRGQ7cjJU2Rk9qEStLakSndTtdDQgsCS3ArEIdiGJnfz4nnN3AVJ22vZD"
        "TAHY3fs899y7/Wxb1eg6M42yleZP64y3zVYZ51qtTKW+fbLNg6uzXI+UXS5LU+mJqqyqtOebEX+byvmsLF2q5mutWqcbtdHOZSvs"
        "4JTHM5+5B1WXrVNZtVVrU3l3rHyjM68W1q9VFj5ztc7xRaEedWOW8UNKkdtCp+quzXPsq6YqUxtTmU1Wqjrz+RqLsdMme9Bhn7Up"
        "Cl0prx2WL22DhxAk6FRnzskZCxz/4Ci/fjbOm2olC1I1w+lb5dumUguN1VptbduopalwHmxUbtWmdR5CVT6DcJny1pYqhwnS5LQt"
        "VtpT7zyD1Yzf0kSeH2UN3qTqApK7tdKZ24pdIGGG3SgS9Md/NzzSVjjmaQ0t9KOBMjmsk22DenVjF6Xe8JB11hSputIQWWkHETKv"
        "YfxGY/VC5xmcsXOA6OnSJLmbn8xnyTVcXpfQYOz1s1d0LW0tjsg2uvchnLOn4khEm6hs6XFmafHM/DvzxlajzlzcYWkaKKUL40fx"
        "U53RUbAwbOjaEs9FPfHXtkYUaAeF4KyP1+czla+zaqURUh+zZ/XdkUKwFW4SJJ/IJ9OLk7v5b1fXtx9PPvx2PpvdqN/U/OT2x9l8"
        "ujSlxl93nz+eXn+YVlTmN3V7fT2/Pzv5dDeT9xPR92m9xavZz5fns6uz2dQuELyPuoBLcs8Xf7uZnc1n59OFXmePBsIFXZbmGW8v"
        "Lv827eIwyEsRZnfzaZqmyRzR+4AoM3oJZ9hKNbb1PNN5XSP8S4sFTXC0t9jA2uisxiJggu/6EzOIBKmLGKPB4zCkp8O5SKJ7hEdj"
        "xHWh2ipIBFWwzI24U9FuamhsSx0epuovWtcKYtUtEsVBj3Ki1lg+QsAYuHqFcFfjzRH3zYp7sWsjjgEYqB8OJWgQUXNGR3ArggUh"
        "8Oe766tJCKmYZ25tnxhKiyx/cGXm1uNKYv7X1kKll2/kWadkbmsYaE3jMMJrTUSAwSQp7RN2aVbtRgukPDXGE8fgkP2zeY5EejwN"
        "KcAIulODaPVlYzd9oh3TiIXO8dcXGRiDcpgw9CbIJ0aJ224WFrbSTQMbE2waoCWVORZUtQsEDpDPbGokLeSUZDkO5g4JxHhKkxDK"
        "yKzNwqzar5eMYEPYtoCHylo3+JP2yUJa6gY+zu0GeGgWpgTuqC5m04TZMVF5Y50bb2zRwovdyxF0R34Qy2lW3QDk3P7WVWEKxthL"
        "WfjJtsq/zfhf5Z8Mn2CJwztYPUP0MegkmjXwqmhzLlOF1U7w0NtWcBvp0BL2PfQiykug7gCEiRYCP03ugKBe0fTKLPe+4WFYBPAQ"
        "h8ATJVwarAk071DRxtQLGWcFtxuzYg5mC/uoJUoEK4wnDgrUoOjYpVerFhHgApIHHHeTIAmWtl69ORQnhEg7YnEaRQHiB2+P5NX3"
        "4ZWgVVsz7d/vr8Q2eI/ofKM+XZ3PbiHC1Xkye2ZE+Qks+WuLjKelJBBhqw6vHql8NOPOtawOLFMOG0gpHYSQpZb0oFlVGeocf0ew"
        "xy/9nOtanOW3SLbhCKn0wHri2oXb4vjNSFyCdSpHugJGzq1YhmivtqhxSfJWfbg+O/lw+fcZcqw0m0WwesZi1/CfSutCF8gCb+td"
        "IdhDProT6LYoUSx1MYRJ+rxiiAgUUY+iq1kscam6bWMFq0xdaxit3gI7qndqrP74x4ObzwcK2KxuPkPCYAumtjdLoyV3oxkmEflA"
        "AcZjU+VlW+jp6xRABDBUr8JKVs1XKgXmEzLV+O2hGpSxQkNKlk5RGsEKkhGidJi8Q25lTb6+d2Zjyqy5Z9QP4NgGTOFhejSc7Nd+"
        "OSgsDcCKRGuXS5MbSE2Bu0T9Ij2BK5aMC6Y04Aqp+isPAHPqC5kUVKkvhObGS/YRDFbcJVWXS+ZBo8mDQGZsi/MO5fsqQN0oEjh8"
        "BX+2VfaIoIC/9I5x0SXCFrEN1aNfxbBPBqSPJb9RD3oroqTJdwgngLHzgbj1PijVq4Cxr/adwW/EIzsHHCVHHRbuOfAVo2Tw6ktH"
        "7vstea+Q0uKJ+0qb1XoBcw8qgSMNPnfPTECOZM+719M3h52rugqNTMuhqY4lgSH8UKFCpYpbwfiF9UxQYPRaLVpT+lB4ApSVOrwY"
        "1A+rNKB0eib5tdEI4WJ4vJMFuZgLa0aOfbgjoFxcXs3O7y+vRury48317fwO5RO5tITfiklk9/BBsDxFbZqAFrSOAHUlBZiYRNOl"
        "yfc7myDzV01Wr8Uk7kWEUvZFuwrFBdgeVJfACjrITxgBTlmbmjYBNSmRAseBir/liUdqD5hY2VVWknGgBcgedZr8IJxXq4HTWt3O"
        "bm6vzz+dzYaomYEisJLqQNvJRGzV/UVMSJJ+hQKeKvw+nSUdUISuIMJFV3z2A7FjtUQ3NAmSGS/r1hIavaxuDTaPsPSi/MGAod72"
        "oNVhmjRSQcm18QDd82tUkLmanV/O1SXIhrQ4hlRcqkcsXYAyWUO3ss5IWZVWJhxtJ8nN5/lP11fn11fzX24v57PTz/PZGenzm6+h"
        "8e2fvnmDtCD3U+N3hwlAhciwJ2SMTYbeJrGQAVG9SWmApNDLoMDgdTZSr18/DCeJwv/qBqVncPAT9OB3B8Pj3YapvLwHzucPA5QJ"
        "46fvh7IqAI/iEf1+STgJR8o5yR9eGlcADJCioXJyIcgWMqdvEmPBUpmH+RatcEyrfqJ5hTTLFwFApB4vbVmCtQZbRqIVkrTPW2Al"
        "4qy2UKNvnvZiiWQ9AeaLG38EoUAJ7thkoJgu5pGNzVJoFuWkqt0sNErGjHWVX01ioWtiixn5zteV07U1PUcysN2RhO7kkSgnytJk"
        "0nxI98Ku1VZgVSyreLwXxqz4jCuX6yprjFWDXTmWiB7torfTXEw1TNVJ4Lrgp8x7gYtNRn4qKuMoOHntI4zueN2+c9PksnrElmZF"
        "Khdbp2CS/0tVQ9/bk9Fglz5n2XMKiKN5yGrh1dlLos1PTPVoS9gw7AKQqz0CoAO9iHQ9d30RllVozp913vLgrylvqs7WmshLFj3Z"
        "9d+lXZmc9t9j2o+wfQYmR/K2EIdIRPPsOmvgfBBBeNuwKddfM/JjIdQRG5/EtUAV0dZ1UN6B6gpc043poVSd/l5jHxhymIWgqSxi"
        "9z2Kffdor+Me9S32qO+pR+yfkRzfheQYSIxomatIbQpZO0zOQmzGkgNxvwh3Msmula1bGCVXJzeXI6TXMmM/OiKPhaXgVzGk2HHH"
        "kUPbJlQ3VZ9iB/5faLLkTceUdxggw5VBmDZ1kLnWm6F8HttXykkr+920Cfqx3+ozzJCnneC7J/oSZ3uhR7pGhJY7Ws+UjroJfDWa"
        "wwrL/g9O2mBTCYs14HMkmxWaYyUkvqE2JdEDi6RVjiM7FPcijBO0DyM68S8QQJT6diT/3pM3AYnvBdX4k2DR/4Fqwp+mMmwffVun"
        "+RJG9/Y5PKu3SIp/wbEpUqekWT1SAvx/g1CVHTg5fPkEJ/r1KLBOUDZYEnRGuJYIFuTq4IZqF4ahY5vtpMs7UQTeSBP+lBHGhPZE"
        "0Wm4Yqrejt9Fh8s0I/g1wz/araUuQLLK/Nrql546K2UkwalW3OoxK9tQindJzERHNsZPxoWWKkecsw865p+Ff9jHEP1F6EjSCK7e"
        "sOxm++OLQNq/nJYkoCW9hhHnnAxKnaNweyrbpuf1qDZryQ2ZpiBaum/cU8baB7F2CzubMk6kgGQ+TD1j3phu2JpFgGD8B7BhBy5D"
        "5mMwO2elEzA95D/Zbrq10yB0xTJebVCDB7QokQ7JxW9JI0sOg5kaGyKFjvVQmntOcVwcFwvHcLHWgJR8SXuSJYLgAMzmW+A4Yuwg"
        "4YiZThksgQ8VMBrKTw9avxz/cCA5xUCZHhwMU4bHYJhQiKlqDg4OQiDxbwkmPEmYg+Elf/WPozIuld5qgCVDNZ2qN6MXj5JOjoOn"
        "g/8njKT0wKWcUaM2c718MBwmkYQFG2DZckiSJAuCuQVe6T2Bn131ioPCu7wRDiV/qpWVqwG/qdWjyUgT0CdsNnB3F8Evbg4A8kfq"
        "59nt5cVnMOAMgPukZbaDSnJM3hswybVEpGFyt0Xlft7r3HP1KsJqhjrutihj/8iIOqx4g//mqM45VFY0W4Za6lKc9viPN5N//vNV"
        "JD70enK7V7QnMkdtv+JzsQsJARj6gpG0MawNjGWwnbnQszipnSTebDSZ+ftDSPioAhu/O7mY3ZzMfwIJDw+urlElb+/Az/tH/5Ox"
        "OzXeRMRV3Ggy4dxZjWuYcwKsBEELv3PgExUj9WvU2CLQZY/7yBumc/bR6jX/OVDjZzX+FZ2yX0zDUOBFR/D2CEZiJhfxamXXBrJl"
        "6wbOkTwFahMH1dGUYfgsjqY5F5xoxLrBtioqBBzpeVphHGcKRRgemErGuiF1yO0QWu/V+eXJj1fXd6G7Q693cnmrBr8zNg9cySy3"
        "sd9Fpg6TE/A+0EkVCkPXAvfTKynhMsCi/GVWwIxpchoXhW530jcNkj9xQECuFWl2nAcIMOLQFl9vaROIfxZ3CvyQGUfSPAmRdhxg"
        "9IlNzQsWLj1IpJTY5ByG4ey35PABDA8CCaOMHJF1ktxSlAlTlygwDnj2upOS9UpIV5rMOgU1sho1yE36m4We8TCnAtxJ4dsbIob+"
        "QihVcvHFVLrRK9QPJ0kmb6JqhVkud7WTBwSyKoWpm3cGcJEPpLP6kTllICnZfj+VHAS0ID+BlnmYnbdVE2M3fjbsrBS+jrcEMGwc"
        "QBQynwT3I/ONa9zeNt0VRJCN9z3RhHu3WZ2/YP4c0XVyesK0lkoPIgMUjXUpVvxu4NdQTae76zPpRBtyEhoJsQBd9w4Z9Bd/vNQY"
        "7i6Hah3ZsojVV+SQCls0Y8ie75EwP1/OfkESXVwkK3zCK6UW+DIOCPDNN4pPYQ41vlJp9zclSa5lvFkxgpj3sEYer5YE1gvNVi+U"
        "X1I1lIt4GSWotEffpOtciYi/Q2N5Z/ocygUk/kHdfTr9eDlPztgju3YBPnUfOnwbmlsXc5UDnO7aEkEhwAJWp4ejfmQnF81GxxHB"
        "buoZIkKkve9w7VDiggMpmdVwwUT4OWepDzIa+oWwsjc2tQ/Bsy83CtcNh6N4mRyGobydEIODW3mZD8h9hN2pEGdAMPbuUh0NQzd8"
        "RG9d7fRh6hRxLP+MJtaEDGkbpv8xZ9JURnq3EBirjDcTiDwDAiCX0WLZ8AJmP/10juYu4UwwRki0Zp9G/nfvhDmL470SvcnJVYjn"
        "cCsSuOGbMLUFy/iFlggM/x1vQohuDI/+eqXUSy/tAK9v0BEswm37U9ZUwk+/Fl+aWkTZeDcSia4+YZflTf//IYjkPvqSw22uzDxJ"
        "coVU+Q9QSwECFAMUAAAACAAAAEddUoJQeJABAACRAgAACgAAAAAAAAAAAAAApAEAAAAAYWdlbnQueWFtbFBLAQIUAxQAAAAIAAAA"
        "R10yfAeLagAAAIEAAAAVAAAAAAAAAAAAAACkAbgBAABjb25maWdzL3NhbXBsaW5nLnlhbWxQSwECFAMUAAAACAAAAEdd3SgpM00A"
        "AABhAAAAEAAAAAAAAAAAAAAApAFVAgAAZXZhbF9jb25maWcueWFtbFBLAQIUAxQAAAAIAAAAR13rzkCvxg8AAC0hAAARAAAAAAAA"
        "AAAAAACkAdACAABwcm9tcHRzL3N5c3RlbS5tZFBLBQYAAAAABAAEAPgAAADFEgAAAAA="
    )),
    "candidates/submission-GRAPH-FUSION-V2.zip": ("a9e9b7a333a059dce7ce88e660b80ef651e8015c00614591e6994e261322bbe5", (
        "UEsDBBQAAAAIAJcYSV1bkfAnWgEAAEICAAAKAAAAYWdlbnQueWFtbFVSsW7dMAzc/RXsbqdIG3TwECB7184Cn8RnE5AoVaTz4r8v"
        "7RfgpRvvRJ54J+FCYiFmVJ3hdy5vBx4EC82wUCkY9EZh6djWcN2Uq4T3H0OpifJnw/Qy/Xy+TGzTX7Tp9oLPv6ZoQyKNnZv5xAyv"
        "A8AfidQNWWyfUud3EujUqrLVvh8lcp9BDbspxJWwjdDRyEGVKyfyeWABb1jI/KxWg4ib0ujq9EFxOy6DhrYCSjqKuI5nqY0kKSgV"
        "FOM4QqYPjphHOJ1B7WAdXd+5rFAl73BbfUNbaXf1iOIroSwEeCzFsjwNLGp9i3eH31hi3hJB67U00++6q1F5KmmwWrPOrjJB3yTE"
        "6qFJumPCFK6c6USU2B7o1tnoAd1z8HBs0xPqdinefXq8E4Q9rkG5cMbut6TH3AGCEC/rpXb9n3ahM4PBH56OvJ0WOz/FEfvyxdqd"
        "cGtYWj4i2LHk4R9QSwMEFAAAAAgAlhhJXb+I2aZNAAAAYQAAABAAAABldmFsX2NvbmZpZy55YW1sSy1LzClNLMnMz7PiUlAoycxN"
        "zS8tiS9OTc7PSym2UjA2MAAK5yZWxJfk5+fEJyfm5ABFDRGiQA3xuZl5pSWpQHEzuHBpUR6QbwpUBwBQSwMEFAAAAAgAlhhJXX1b"
        "577IAAAAJAEAABUAAABjb25maWdzL3NhbXBsaW5nLnlhbWxlj8FOwzAQRO/+ipF6RK1KgdIY8S2WSTaOFccbrdct/D1uQFy4zaxm"
        "dvftsPhPx1XXqk55plzwjudjd4ZOMc8xBzz8+JEFypz2vU8JXkJdKGt5g0dqjgRXnyqhTNKKpfXJ7CDMCxKNuvV7zleS4jVyxhSL"
        "snwh5hIHuufxdHo9X/bbH7jFPPDtYJSWlcRrFbI4Hk5GeXXrXXYvm55te9D847C4PHYt/Yvh2u0xBmvwR+Y+6hBI7cbX5jH3qQ7k"
        "dOIaJm0bVCqZb1BLAwQUAAAACADIGEldB53FK7QSAAADKAAAEQAAAHByb21wdHMvc3lzdGVtLm1kfVprc9tGsv3OXzHLSq1ILwlJ"
        "jp1kwSguRoYtVmRSRVGxXI5XNQQGJC5BAMZDEm/px+/p7gEIWs66KhEJDBo9/Th9uoef0krp3CidKF2VaZJu06pQRRqWD3TZJKso"
        "MSZXYfQYJSs1m3oqN1laRGWa71RUFJVRUaKOH9J8U2TaNyNVrk19o+AvVQEBW1MUemUc9RErVRqGMQS7KjH3uBklRanjWEHABosK"
        "leb0lIhKSpMnpnTURRQEJlGlKcpCBcaPAqOuxtfXx+/Gk0tHLdZmpx4MtH7Io7LEyhBiSMRWQwj+M3lxRDsZYIN0Y6d0Ad1KZR61"
        "X6pEQ0nci1aJLqucPgcm1FVc4pPVH5/udVxBRZ0EqtxldIU/krjcDP218bG/OFBLs9b3UZo7nc785tK77pw6akzWi6FmVK5VAiXS"
        "NFY+bd2bvr1mZfMqcZQHiWSZnYIiiSzXrdVVUkaxKqrlNirvMl36a7XWBYTTchOwE5L2u9LEKLH41XhxfuG9hR1i47rFbruE0KF6"
        "WOtS+WudrEzgdF46asqeMUFUisWPB/zX/kEw2E93L5xsN1Av7ugLf/TTJGy+ZDv+GCURLAv1MscPV3gyfZRr2S7L0/8zfumU6TZ2"
        "1LWf83ZWKWxMkVVuM0ed50aXiFLW+jDgsDVsklyt1cokFTaJ74l5UBAcVH4ZpYna4lOM2JRwo/gwdywqpe8IfvMY8ab4BU7nRyiy"
        "haGheluM2AfG1RQzWYyYkhCHGTPo75JPc8SMj/RAYPix0UmVDUi9rS7pBQMVbbM0J1vDVQM1vppYsRz1pHdgMpMg0v3IFI76w5js"
        "b2JS3F1HoXn0TUZqSmwWu8Q/1vT/JhLhmS2CJVqKKWAhnSPmkV8Fg8DG7GDWYEgRM7SvcdQ7Mm0iJqUUGewfI41tYkB2tmNbJAay"
        "inJv0hZgIDTuTUJKStbkpklzQowjuDyg+2HEOmFFnXcKjlpC963TeQXn4B0U8xGZYYUX8Ed+YJXrbI24pWdoU3BBgEVJyp5Mw9GB"
        "khSchFPmnl7sG1nILs2MP+KIe0SirSCuNAGF3gG+abXeZSkuFQR2aUopHuoo5n1X2GWZI0aXmjAhly8qrfA32RRsVfgwjwwwBT7e"
        "RrFGaO6czmuSQ+k+1Pmq2sIkyuQ5JOAtpKDml0Cf/dtdiNJQsBywyqRl/ayECsyfYoeOmoR4vhALQqQYjLGF38FxKKHFL1vTXsqH"
        "iIwT5GmGVzidnxhvgS1JkD6QVj++/AMyNibhrC1T4Dm7gxIsjLEGIV3UAOcCxTILcLBGVpWqVxijzmcfPowBhP2BzdSg2maAphSZ"
        "SonJzoa2+LuMVux41jasgInsyqJCaltIFhFYA+w42DFgmm1VJRbwEJcBsr6zuPDU5Wx25dbg2wDutoLw3AAIDGMpRQei1CcZqCBU"
        "YHadi30oDH9TOSEWYaFEFl1CcdAZqakFTxhI/LRCtbDQoukxMjhWp0sUp3t+kDQmcfjIiIxNmAKK2WtQNgp3stBnSMNHqQ4WQMiL"
        "G5JbrAl8CC5zWD6irZSw4oh2lFD4wNxUQxtFsBm4z+nM6V1hWgEMHpFal7OPA/XBezu5+UDaXEzeX6jeJS4N1EXfqe+cKYBkVRDi"
        "SGbZPQUOCcDtKtkk6UOiGJwS5EUQ+SXdZoHW6vgfjAWUlqc1224H5uLjmg+Hj1QU1hfoLXSNMyEq64wheU5HLVSJrDCli21GiAZb"
        "//hF4gJ5tasmdT5xLOGPMA6J4RpRB8pSDAEELkL7jGdCwemi5jA3KQZjGHr37kAMKfmQp8lq/3LaGj3P6xiJd1zAraUQBBHbhJkH"
        "ruTwfLnnHOoyTTfWlbSApQt1IQm0l4IEpHkEhqc5Z9g/VEzlMbEdgXdOHumoW0C68St6sSuhK7oJ2hW+SYBeVP+QMDtKc7IVomy1"
        "5oVi+HqH+hAm22br6Rj8jLLsa5V+C7r9Zu2h7VE6KI44WVcpK4RV0WpdEoNMxQtXirlSS3ubdkBM9hSTJeSFWmIXnC9tJjfinA6U"
        "Dqn20eOUjPWO2sYAodxmsWn8E0TsHoaoDKTT6byLcnzeIlmRVDEyrBAKtg+mVXRvCrdtmUGrOrZLL5J9Bd25DkYlFS3YNNFIukYR"
        "goYq4btN0LGvCWcIBMxWQ7IPkwP4GNwRXfGD3kEvZuziAyjK6RiqxUDNB+pWoHarN4a4D5sfeI2ijrRF0DYZeQhvTud6MV54dYfA"
        "miB6ckntJSC0zrQQ0Rlb6nofMZLAjI+lXXTAisUvO8KokM3bwgsqGQawZxOKb5P3BvYpAXxbVS2ewkat6l4gNvy17FcvKUjkmdPX"
        "/HZQtRnwtlVU4KY4KFy71cXZhZqffVC3Z5fq6myontR4eu7a1qPNep7UxSdUoGyzOgZtBYl23RBCBwIe9NzlwlWkVU7WDaKA0hfX"
        "vT9dRj68GTLpCnN9V0onvi68azwZawFV8s6Tmnq3C4ldMaEN2Br8S5u2RWdOtBrOsY5Dx0bwiECw1bROFqnn1PTMbhYesZKh7dXa"
        "hm55pqM8Kmkrqi9glaq3NIiyvmvdI74YNOVsAgYJXsnLa3ayhV8Z3LmYIDDJdbcqSKtlCW4As6xTiklqX9PQMiXNOzbAcMqM72Tv"
        "4O8zw9Zjfmf9FupvoKyOW15x949nMRrrGKQYbx1Ifwa73hHzuEsMgGrJ1MummTWolKeREHcbu3vpeP11/UpQO3A2Y58oxBKNO6F7"
        "walFhcbW9+fFIGQY6kE2bJOn6I0L2zkheyAO0bmOMpKBeISK+eBZbdnq3b6mqCoDCBu9hTObvcKPzOGczkdqTMuHlHdnON7qdMOL"
        "g5QLEvxMgU4edvcFscDi2HyfTxWG+pNS8GOren/rXY4CcnEdB/SZIFo+EY/KEbfyrd4KjQzQjBGt5gK0mI/PvT4bOkgFyEgLIpOz"
        "2eU1oliDZuikeOCOBpUgDGF/EPraJx3LTO9sA8Bmcq2bgpT3AZPvYTunKkZDEcDivXmDTuHH4S+E0gGHAZpAla1zjXgvKnp3obpN"
        "S7lrtkUQkMZczbs13ZbCgzapMNSAFIaYhgS+bA00aw+fdY9As6Mt+HjFoyNDhbYuR9/Bt1aj2hqq4EVFqblV2TelMmCo5zJ7goDt"
        "ryn4n+cPmU2qkLWapMNAaisbMyr5/hs1NSYoxP+HydOWXEfrc398P33gjWJjpxHFWlNhp+7fB3JTtxtLLUIW1SbXwGMArPQ1I7Tg"
        "TNYNXPqa1qMzRKhQ+eO+isGXUIeizq17AG6nrbiqTLe44jvqpjCWzifgQnRN1ThdUFkfq2VV1h0Z4hKeyA0PwL6FMUX1RyYLELRr"
        "c3pLwYqKGiIuPXYpIwElsHBG6q7QEizU+zEVBW5iWDOiEZw9tzKHRA0iOsWSeYF+Rqp4sCDN/oL/P5dHS4JNxIvtPXo1QbMsQ1DA"
        "PGYoKJS+VAbuafpHK0J7U97AXiblwzjK+tLqSt2zrOOQQwhVU4vx/L2H/ZyPb669ARVftERrtL4TRmN6D5FiAQvv9so7X6jewehx"
        "nw/7IY8dNzKD5sknGVbmln1xMqkg0yw7J5Qer0qirxShcXAH6OLZVp5upbxoTiwKhxGVHLJwlZumMT49sSEnM5r96OCKZ3PlA829"
        "eq0JU6Yjws8GMYMqi1HoKCDWJobF++LM1qxM6QxraGepWgL0ESB/evPJu09gWIEGPtwDBCG81I+j73LrHo/xOIHhPpDPNsnmLrhm"
        "kI/14E84BO1YBCPcGNsTVLjCNlKBqTkrnNUffUvfRzw6Wu6sM+p+KaFsz+0koZ6jwvu8efHLPq2o2hAfJYj2QccY/2B+IF/BNAqa"
        "Ikh1HP0/Jze0oLKB5OZ2Y6xQK3c1pRbdqOEyMqk0NPkgaINJqoQxB5dYXUed83h6il0Dh+GH3T7UZFxDowhKQppYFu12sNUtlSnq"
        "iq2vjvpYO5bVo2TjsaeIaY+ov/ECTV1JA9mbHTKy82j/dgxse3KwSe98hvhwLYvU31D1hhlYe3K+I7PAkGy4NbzcFfQUp9QY0UIu"
        "dJIkzVHT2WI4v5kKO5bRHTKrbtRsmwuZQWzzs26BqQMjfORxLyULs1fWjPYm1H2Z64TGUaE0LEKzyXof57Pp+zvv1ju/WUxm0zvB"
        "ERnyM8Gh8oc2eO9bjrnmSYEh8B4zjFOfgsjUAShUdLCnoU3hBJTMx9iqiJjPZos7hjHV2+5k1J1Ib/ag7ajCVVyLy7RFSC2n4kOf"
        "53yp7+y9UWcIlSCmeSk1eJPp+ezD1aW38O7eTW6d7xLk/HlxpspIEGJzgQrVSMlBjEAVR7ZtmDj5oMlYYs1atKD6bBtEwmqKLrAJ"
        "Ume8mPw+uZwsPt3Nvfdz7/oaToG1aKPsOUlNBsqirBvM9tigOYc6SC6oMOGTgAG1BisKEnIsV1TUJ+4veeSLT1xz+GXrKtm4ypv+"
        "OYGjPnjTxR0dft3MPUptTk4qODQKk3Z/P8+WEefYevhFu7ON6DiNOqLW0JSm0kiVQt8bZnO8y+FQvZtceuo3PpY5Xk8dDstBXUk5"
        "zGg1G5+2YJ9AadP5RqZoOmiYjByMHLBixAGy/febt4hhF4XoZN/fI7x+OlHbKKlKW5pe023hoDn1Ho/RttrSbFmjD46amdaSzpeo"
        "RDcIxLNOaQDhJmmsBbv0PTxPoxNU/V1GWSKrrSGlGVa9MEqignvPIoXKPMGzQzg0O974+hN0H748qRWfzuYfxpfq5cnw1euBuhjP"
        "36pXr4c/nwxABBZz74Onfj4Z/vtE9XQIkwQ8uxHw5dM77qkLAguZPNQFGml7zhMvpARxvKrgQt2aULUnHQKKrdHFy2Z0cUXE6ZcT"
        "hWrlsxtCy6JtOALByjRjJEiZS/ARkwC5TDLxPGMxwobppqHRVZImQwH55p4A2dJorjiQuIb1S4aCgOaDnU59CiCtGxsAH5DyW+AX"
        "CMvV7HpyCyi2bLl1FMiFZ8QgdECIeHCac3oQe6JzUVpO6/oduBKB3gzQhokaTtTwnRoadTSenl/M5kcUxkd0znmknkBoEMFw7UgZ"
        "H81G9zOi40u3wwW2LSd+LoHWHIr5qUNK0JggoDcfjQe/Z0eSZE/KpwTyT4env5x05NQkODhpv/q0uJhN386mi4/zycL7/dMCRfKt"
        "d3Zq71DlONs/4O4/Hhe538DMKcI025XrNPlR8pqZBNSs8zx3aNnL3/55ut/zY1Se/fDmSxcECQkDe7xure0IvXIbsUNfde25JwJt"
        "UOyQip81HRGD8pheigzrgWiTLXoobVmfMTMjoMBaB1l3//nU/fJlhOY0SsrekeVv6eao32VzXbMD3P9hk2a7r/fbHW7tATXPxliO"
        "Gj6q4Vf4q1yeCZEeZohj10cfT3ahg8KcjQEP1XvvMPd55qDi7IceBYRNzeGQBfZrK/5QwHwtcKVljdDXrVVNLLyiuaGElxepo95/"
        "no77PY68N0/2WL5//MSX+Wz+83+Ov7z4C6784YnP5+VjfUBP355k/3/RafwTH9D/RSf0T2X6KNeaA/q/+IT+qAmBVaXzgGOgM015"
        "1iWZxGeEyB9pgnOzMo+OkijXNPGGsX+28OXIYWBm+9Ca2dt5f0GcTuolgxgnM5qKJizrMP311yNv9u5IOY6j8GFw+LMOaaAGckx+"
        "cKY8kGhqDqXUfcH9oZxASTyhtbARElCzotmZOczER1K2dKNjCWyYXF3evJ+AH9iAeKD2HUScSzsfT7PVTokivnkjwMSHT0v+DQ79"
        "2CUJCCX3PzogIBvVFZZo6IYRNufR/PcKLqW90xGSCzlL47atxldaVuvYvESeUYWwI4LOXWFQLM7U5y8dNAjqbp2mm16Y8/m/uefp"
        "F7Ky73YU/sFIfE2dnakueawr1+mfDyH8nBMya2tuhLjhO37KSE2Mu7mzwZ1eOKCqpssy74GxdrHua6VjWtcdyHP0ud9vnoISzWX1"
        "Dyjyq8wuf+uy4UOHHUdT/HWv24LCrjTk3WPycpfLPDwqlK5boDUY1j9L+ubmpv7KttrvmP7xJQctLtzZ2/TZhvz+njUZwRrSDd6g"
        "7ffYvtxtWgc8v2llpFnvBY8Lvi+IWjuxCeUfRZQMF/bqoQ7Cvmi2el/JX3z2+tWh43kxjdNV/1KJIPDdQH1tdrg3tYBwFx1Dl8xO"
        "Ek0MCtW1LRMNFAccuGesXgnIzPdahW2pn91XJ1/cb0Vz9EJM+PkUuI/13xGH6O3YIVeUqBoN6h/Y8K8W1qb+TZnFELQrTBps0Cdp"
        "VPAPMkZi2b68v6BmiOxi90O0yP72pRkkibxXJ3xC2pqQ2XlX4LYSa8ReIgwApBI09UD8upSOXfjcruPErB+6Gp//MX7vdSRbbeR0"
        "ynwnlppcX994oI5j6pY6fEYW21v1E9hNt+nM7mr9YNLuOTWiztagBAbdfue/UEsBAh4DFAAAAAgAlxhJXVuR8CdaAQAAQgIAAAoA"
        "AAAAAAAAAQAAAKSBAAAAAGFnZW50LnlhbWxQSwECHgMUAAAACACWGEldv4jZpk0AAABhAAAAEAAAAAAAAAABAAAApIGCAQAAZXZh"
        "bF9jb25maWcueWFtbFBLAQIeAxQAAAAIAJYYSV19W+e+yAAAACQBAAAVAAAAAAAAAAEAAACkgf0BAABjb25maWdzL3NhbXBsaW5n"
        "LnlhbWxQSwECHgMUAAAACADIGEldB53FK7QSAAADKAAAEQAAAAAAAAABAAAApIH4AgAAcHJvbXB0cy9zeXN0ZW0ubWRQSwUGAAAA"
        "AAQABAD4AAAA2xUAAAAA"
    )),
}

for rel, (want, b64) in EMBED.items():
    dst = (TOOLS if not rel.startswith("candidates/") else WORK) / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    data = base64.b64decode(b64)
    got = hashlib.sha256(data).hexdigest()
    assert got == want, f"embedded file corrupted: {rel} {got} != {want}"
    dst.write_bytes(data)

CANDS = {  # label -> (archive path, role)
    "VNEXT-ORIGINAL": (WORK / "candidates/submission-VNEXT.zip", "NEGATIVE CONTROL (old mirror: include error)"),
    "VNEXT-INCLUDEFIX-ADDITIVE": (WORK / "candidates/submission-VNEXT-INCLUDEFIX-ADDITIVE.zip", "repair under test"),
    "EVIDENCE-V1": (WORK / "candidates/submission.zip", "control (assumption A1)"),
    "HYBRID-V2-ADAPTIVE": (WORK / "candidates/submission-GEMMA-HYBRID-V2-ADAPTIVE.zip", "control"),
    "GRAPH-FUSION-V2": (WORK / "candidates/submission-GRAPH-FUSION-V2.zip", "control"),
}
auto = []
for p in glob.glob("/kaggle/input/**/*.zip", recursive=True) + list(EXTRA_ZIPS):
    try:
        with zipfile.ZipFile(p) as z:
            if "agent.yaml" in z.namelist():
                auto.append(p)
    except Exception:
        pass
for i, p in enumerate(sorted(set(auto))):
    CANDS[f"EXTRA-{i}-{Path(p).stem}"] = (Path(p), "user-supplied archive (e.g. S0)")

def safe_extract(zpath, dest):
    dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zpath) as z:
        for info in z.infolist():
            target = (dest / info.filename).resolve()
            if not str(target).startswith(str(dest.resolve()) + os.sep) and target != dest.resolve():
                raise ValueError(f"unsafe path in archive: {info.filename}")
        z.extractall(dest)

DIRS = {}
for label, (zp, role) in CANDS.items():
    d = EXTRACT / label
    if d.exists():
        shutil.rmtree(d)
    safe_extract(zp, d)
    DIRS[label] = d
    RESULTS[label] = {"role": role, "archive": str(zp), "bytes": Path(zp).stat().st_size, "sha256": sha256_file(zp)}
    print(f"{label:28s} {RESULTS[label]['bytes']:>7d} B  {RESULTS[label]['sha256'][:16]}...  {role}")


In [ ]:
# ---- cell 6: compile_submission preflight per archive (installed wheels; stub model unless the registry helper exists) ----------
def run_json(cmd, timeout=900):
    p = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    try: return json.loads(p.stdout), p
    except Exception: return None, p

for label, d in DIRS.items():
    rep, p = run_json([sys.executable, "-I", str(TOOLS / "preflight_compile.py"), HARNESS_DIR, str(d)])
    if rep is None:
        rep = {"VERDICT_compile_submission": "NO-JSON (preflight could not even import the harness)", "stdout_tail": tail(p.stdout), "stderr_tail": tail(p.stderr, 3000)}
    log(f"20_preflight_{label}.json", rep)
    RESULTS[label]["preflight"] = rep.get("VERDICT_compile_submission")
    RESULTS[label]["preflight_notes"] = rep.get("notes")
    fails = [(s["step"], s.get("exc_type"), str(s.get("message"))[:300]) for s in rep.get("steps", []) if not s["ok"]]
    RESULTS[label]["preflight_failures"] = fails
    print(f"{label:28s} compile_submission: {RESULTS[label]['preflight']}")
    for f in fails: print("      FAIL:", f)
    for n in rep.get("notes", []): print("      note:", n)


In [ ]:
# ---- cell 7: FAKE-MODEL REPLAY (NOT A GEMMA TEST, NOT A SCORE TEST) with real LiteLlm objects + request capture ---------------------
for label, d in DIRS.items():
    out_file = LOGS / f"30_replay_{label}.json"
    p = subprocess.run([sys.executable, "-I", str(TOOLS / "replay_http_capture.py"), HARNESS_DIR, str(d), "--out", str(out_file)],
                       capture_output=True, text=True, timeout=900)
    try: rep = json.loads(out_file.read_text())
    except Exception: rep = {"VERDICT": "NO-JSON", "stderr_tail": tail(p.stderr, 3000)}
    RESULTS[label]["replay"] = rep.get("VERDICT")
    RESULTS[label]["replay_checks"] = rep.get("checks")
    RESULTS[label]["replay_error"] = (rep.get("exc_type"), (rep.get("message") or "")[:300]) if rep.get("VERDICT") != "RAN_TO_COMPLETION" else None
    RESULTS[label]["thinking_fields_per_request"] = [c.get("thinking_related_fields") for c in rep.get("http_requests_captured", [])][:6]
    print(f"{label:28s} replay: {RESULTS[label]['replay']} | checks: {RESULTS[label]['replay_checks']}")
    print("      thinking fields in requests:", RESULTS[label]["thinking_fields_per_request"][:3])
    if RESULTS[label]["replay_error"]: print("      ERROR:", RESULTS[label]["replay_error"])


In [ ]:
# ---- cell 8: does the INSTALLED resolver honour thinking-off settings?  (variants of the additive VNEXT; test copies, not candidates) ------
base = DIRS.get("VNEXT-INCLUDEFIX-ADDITIVE")
VARIANTS = {
    "BASE_include_thoughts_false": None,
    "V1_thoughts_true_budget4096": "  include_thoughts: true\n  thinking_budget: 4096\n",
    "V2_thoughts_false_budget4096": "  include_thoughts: false\n  thinking_budget: 4096\n",
    "V3_thoughts_false_level_none": "  include_thoughts: false\n  thinking_level: none\n",
    "V4_budget_zero_only": "  thinking_budget: 0\n",
}
VAR_RESULTS = {}
if base is None:
    print("NOT TESTED: VNEXT-INCLUDEFIX-ADDITIVE not extracted")
else:
    for name, tc in VARIANTS.items():
        d = WORK / "variants" / name
        if d.exists(): shutil.rmtree(d)
        shutil.copytree(base, d)
        if tc is not None:
            for f in ("configs/sampling.yaml", "sub_agents/configs/analyzer_sampling.yaml"):
                s = (d / f).read_text()
                (d / f).write_text(s[: s.index("thinking_config:")] + "thinking_config:\n" + tc)
        out_file = LOGS / f"31_variant_{name}.json"
        p = subprocess.run([sys.executable, "-I", str(TOOLS / "replay_http_capture.py"), HARNESS_DIR, str(d), "--out", str(out_file)],
                           capture_output=True, text=True, timeout=900)
        try: rep = json.loads(out_file.read_text())
        except Exception: rep = {"VERDICT": "NO-JSON", "stderr_tail": tail(p.stderr, 2000)}
        seen = sorted({json.dumps(c.get("thinking_related_fields"), sort_keys=True) for c in rep.get("http_requests_captured", [])})
        VAR_RESULTS[name] = {"verdict": rep.get("VERDICT"), "thinking_fields_sent": seen, "error": (rep.get("message") or rep.get("stderr_tail") or "")[:250]}
        print(f"{name:32s} {VAR_RESULTS[name]['verdict']:20s} sent: {seen}  {VAR_RESULTS[name]['error'][:120]}")
log("32_thinking_variants_summary.json", VAR_RESULTS)
print("\nRead: enable_thinking=false  => thinking really disabled on the wire; true/absent => the setting did NOT disable it.")


In [ ]:
# ---- cell 9: what do timeout_seconds / command_timeout_seconds mean in the INSTALLED code? ----------------------------------------
hits = []
roots = {Path(adk_submission.__file__).parent, Path(swegemma.__file__).parent} if IMPORT_OK else set()
try:
    import adk_eval_core; roots.add(Path(adk_eval_core.__file__).parent)
except Exception: pass
for root in roots:
    for f in sorted(root.rglob("*.py")):
        try: lines = f.read_text(encoding="utf-8").splitlines()
        except Exception: continue
        for i, line in enumerate(lines, 1):
            if re.search(r"timeout_seconds|command_timeout|max_exec_seconds|test_timeout", line):
                hits.append(f"{f}:{i}: {line.strip()}")
for extra in disc.get("scorer_script_hits", []):
    if Path(extra).name in ("inference.py", "metric.py"):
        for i, line in enumerate(Path(extra).read_text(errors="replace").splitlines(), 1):
            if re.search(r"timeout|max_time|max_tool|eval_config|thinking", line):
                hits.append(f"{extra}:{i}: {line.strip()}")
lim = {}
try:
    limits, gc = importlib.import_module("swegemma.config").build_submission_limits()
    lim = {"limits": repr(limits), "generation_constraints": repr(gc)}
except Exception as e:
    lim = {"error": f"{type(e).__name__}: {e}"}
log("40_timeout_grep.txt", "\n".join(hits) or "NO MATCH")
log("41_build_submission_limits.json", lim)
print(f"{len(hits)} lines mention timeouts; first 60:"); print("\n".join(hits[:60])); print(json.dumps(lim, indent=1)[:1500])


In [ ]:
# ---- cell 10 (OPT-IN): real Evaluator on ONE task with the scripted fake model - exercises the real agent_runner compile path -------
# Needs the competition data mounted and a working sandbox.  NOT TESTED by the author (no access).  Set RUN_OFFICIAL_EVALUATOR_SMOKE = True in cell 1.
EVAL_RESULTS = {}
if not RUN_OFFICIAL_EVALUATOR_SMOKE or not IMPORT_OK:
    print("SKIPPED (flag off or harness import failed) -> NOT TESTED")
else:
    import asyncio, concurrent.futures, yaml
    sys.path.insert(0, str(TOOLS))
    import replay_http_capture as R
    from google.adk.models.lite_llm import LiteLlm
    from google.adk.agents.context_cache_config import ContextCacheConfig
    from google.adk.apps._configs import EventsCompactionConfig
    from adk_submission import ModelRegistry, discover_adapters
    cfgmod = importlib.import_module("swegemma.config")
    from swegemma.evaluate import Evaluator
    from swegemma.models import load_tasks
    srv, url = R.start_server()
    tasks = load_tasks(DATA_DIR / "tasks.jsonl"); task = tasks[0]
    print("task:", task.instance_id)
    def run_sync(fn, *a, **k):
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(fn(*a, **k))).result()
    for label, d in DIRS.items():
        try:
            ev = yaml.safe_load((d / "eval_config.yaml").read_text())["evaluation"]
            models = ModelRegistry(); models.register(MODEL, LiteLlm(model=f"openai/{MODEL}", api_base=url, api_key="EMPTY", num_retries=0))
            limits, gc = cfgmod.build_submission_limits()
            adapters = discover_adapters(str(d), adapter_extensions=cfgmod.ALLOWED_ADAPTER_EXTENSIONS)
            want = dict(tasks_path=DATA_DIR / "tasks.jsonl", snapshots_dir=DATA_DIR / "snapshots", results_dir=WORK / f"eval_results_{label}",
                        submission_dir=d, models=models, sandbox="subprocess", timeout_seconds=int(ev.get("timeout_seconds", 300)),
                        max_time_minutes=float(ev.get("max_time_minutes", 5.0)), max_tool_calls=int(ev.get("max_tool_calls", 100)),
                        max_turns=ev.get("max_turns"), limits=limits, generation_constraints=gc, adapter_manifest=adapters,
                        context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
                        events_compaction_config=EventsCompactionConfig(compaction_interval=15, overlap_size=2, token_threshold=14336, event_retention_size=5),
                        graph_dir=str(DATA_DIR / "graphs"), embeddings_dir=str(DATA_DIR / "embeddings"), wheels_dir=DATA_DIR / "wheels", verbose=False)
            accepted = set(getattr(cfgmod.EvalConfig, "model_fields", None) or getattr(cfgmod.EvalConfig, "__dataclass_fields__", {}) or want)
            dropped = sorted(set(want) - accepted)
            cfg = cfgmod.EvalConfig(**{k: v for k, v in want.items() if k in accepted})
            R.CAPTURED.clear(); R.STATE.update(calls_by_agent={}, n=0)
            res = run_sync(Evaluator(cfg).evaluate_task, task=task, task_index=1, total_tasks=1)
            fields = {k: (str(v)[:300]) for k, v in (res.model_dump() if hasattr(res, "model_dump") else vars(res)).items() if k != "agent_patch"}
            EVAL_RESULTS[label] = {"dropped_kwargs": dropped, "result": fields, "thinking_fields": [c["thinking_related_fields"] for c in R.CAPTURED][:3]}
        except BaseException as e:
            EVAL_RESULTS[label] = {"error": f"{type(e).__name__}: {str(e)[:500]}"}
        RESULTS[label]["evaluator_smoke"] = EVAL_RESULTS[label]
        print(label, json.dumps(EVAL_RESULTS[label], default=str)[:900])
    srv.shutdown()
    log("50_evaluator_smoke.json", EVAL_RESULTS)


In [ ]:
# ---- cell 11: summary + logs archive -----------------------------------------------------------------------------------------
vers = disc["versions"]
lines = ["# PREFLIGHT SUMMARY (current wheelhouse as mounted in THIS notebook)", "",
         f"swegemma={vers.get('swegemma')} | adk-submission={vers.get('adk-submission')} | adk-eval-core={vers.get('adk-eval-core')} | google-adk={vers.get('google-adk')} | google-genai={vers.get('google-genai')}",
         f"python {sys.version.split()[0]} | install mode: {INSTALL_MODE} | harness dir: {HARNESS_DIR}", "",
         "| label | role | bytes | sha256 (16) | compile_submission | fake replay | evaluator smoke |", "|---|---|---|---|---|---|---|"]
for label, r in RESULTS.items():
    lines.append(f"| {label} | {r['role']} | {r['bytes']} | {r['sha256'][:16]} | {r.get('preflight', 'NOT RUN')} | {r.get('replay', 'NOT RUN')} | "
                 f"{'ran' if 'evaluator_smoke' in r and 'error' not in r['evaluator_smoke'] else r.get('evaluator_smoke', 'NOT TESTED')} |")
lines += ["", "Thinking variants (what the installed resolver put on the wire):"]
for n, v in VAR_RESULTS.items():
    lines.append(f"- {n}: {v['verdict']} -> {v['thinking_fields_sent']} {v['error'][:100]}")
lines += ["", "Labels: a PASS here means ONLY that the installed compiler accepted the directory (and the replay ran with a scripted fake model).",
          "It is NOT a Gemma result, NOT a score, NOT the scorer's own inference.py/metric.py."]
text = "\n".join(lines)
log("99_summary.md", text); log("98_results.json", RESULTS)
print(text)
archive = shutil.make_archive(str(WORK / "preflight_logs"), "zip", LOGS)
print("\nlogs archive:", archive, "(download this from the notebook's Output tab)")
